# TGDetect: training length, fast counts, and rarity robustness

Upload to Colab, choose GPU (T4 or L4), then Run all. Original versus frozen
log-count FiLM only. Dataset download and source code are bundled. No API keys.

Two encoders × five seeds × two new split seeds = 20 training trajectories.
Each trains to 16 epochs; report validation-selected checkpoints available at
8/12/16 epochs. This gives 60 budget evaluations without restarting training.
The first two studies remain historical; this notebook writes a separate root.
Exact input deduplication, benign-only learning/calibration, and disjoint benign
reference counts follow the previous protocol. New random splits may overlap
earlier data and do not constitute independent captures. KDD is dated, simulated,
and not a full TGDetect evaluation. No labeled Back/Warezclient training is added.


In [ ]:
import importlib.util,subprocess,sys
packages={'numpy':'numpy','pandas':'pandas','sklearn':'scikit-learn'}
missing=[install for module,install in packages.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import torch
assert torch.cuda.is_available(),'Select GPU runtime.'
print('GPU:',torch.cuda.get_device_name(),'PyTorch:',torch.__version__)
print('Free GiB:',torch.cuda.mem_get_info()[0]/2**30)
assert torch.cuda.mem_get_info()[0] > 4*2**30,'Restart runtime to free GPU memory.'


In [ ]:
from pathlib import Path
import sys,json
BUNDLE=Path('/content/tgdetect_stability_bundle');BUNDLE.mkdir(parents=True,exist_ok=True)
SOURCE_BUNDLE={'orthrus_comparison/universal.py': '"""One shared, bounded-vocabulary encoder for unordered typed key/value sets.\n\nNo source-specific learned encoders. Numeric features supplement shared subtoken\nembeddings; missing and nonfinite values get explicit types. No field positions.\n"""\nimport hashlib\nimport json\nimport ipaddress\nimport math\nimport re\nfrom functools import lru_cache\nfrom collections.abc import Mapping\n\nimport torch\nfrom torch import nn\n\nKINDS = (\'missing\', \'bool\', \'number\', \'ip\', \'path\', \'text\', \'list\', \'object\')\n\n\n@lru_cache(maxsize=262144)\ndef bucket(text, size):\n    return 1 + int.from_bytes(hashlib.blake2b(text.encode(\'utf8\'), digest_size=8).digest(), \'big\') % (size-1)\n\n\ndef value_parts(value):\n    if value is None:\n        return \'missing\', [\'<missing>\'], 0.\n    if isinstance(value, bool):\n        return \'bool\', [str(value).lower()], float(value)\n    if isinstance(value, (int, float)):\n        try:\n            number = float(value)\n        except OverflowError:\n            number = math.inf\n        if not math.isfinite(number):\n            return \'missing\', [\'<nonfinite>\'], 0.\n        return \'number\', [\'<number>\'], math.copysign(math.log1p(abs(number)), number)\n    if isinstance(value, Mapping):\n        # Leaf dictionaries are normally flattened before this point.\n        return \'object\', [\'<object>\'], 0.\n    if isinstance(value, (list, tuple)):\n        parts = [\'<list>\']\n        for item in value[:32]:\n            kind, words, number = value_parts(item)\n            parts.extend([kind] + words)\n            if kind==\'number\':parts.append(f\'<signed_log:{number:.6g}>\')\n        return \'list\', parts, math.log1p(len(value))\n    text = str(value)\n    try:\n        addr = ipaddress.ip_address(text)\n        bits = (8, 16, 24, 32) if addr.version == 4 else (32, 64, 96, 128)\n        return \'ip\', [str(ipaddress.ip_network(f\'{addr}/{b}\', strict=False)) for b in bits], 0.\n    except ValueError:\n        pass\n    parts = re.findall(r\'[\\w]+|[^\\w\\s]\', text[:4096], re.UNICODE) or [\'<empty>\']\n    kind = \'path\' if \'/\' in text or \'\\\\\' in text else \'text\'\n    return kind, parts, 0.\n\n\ndef flatten(attributes, prefix=(), depth=0):\n    for key, value in sorted(attributes.items(), key=lambda p: str(p[0])):\n        path = prefix + (str(key),)\n        if isinstance(value, Mapping) and value and depth < 8:\n            yield from flatten(value, path, depth+1)\n        else:\n            # Length-prefixed components prevent collision between nested and dotted keys.\n            yield \'\'.join(f\'{len(p)}:{p}\' for p in path), value\n\n\nclass UniversalEventEncoder(nn.Module):\n    def __init__(self, dim=64, buckets=16384, max_fields=64, max_subtokens=32):\n        super().__init__()\n        if dim<4 or dim%4 or buckets<2 or max_fields<0 or max_subtokens<1:\n            raise ValueError(\'dim must be a positive multiple of 4; buckets>=2, fields>=0, subtokens>=1\')\n        self.dim, self.buckets = dim, buckets\n        self.max_fields, self.max_subtokens = max_fields, max_subtokens\n        self.tokens = nn.Embedding(buckets, dim, padding_idx=0)\n        self.kinds = nn.Embedding(len(KINDS), dim)\n        self.numeric = nn.Linear(1, dim, bias=False)\n        self.pair = nn.Sequential(nn.Linear(3*dim, dim), nn.GELU(), nn.LayerNorm(dim))\n        self.attention = nn.MultiheadAttention(dim, 4, dropout=0., batch_first=True)\n        self.query = nn.Parameter(torch.randn(1, 1, dim)*.02)\n        self.output = nn.Sequential(nn.Linear(dim, dim), nn.LayerNorm(dim))\n\n    def tokenize(self, events, device=None):\n        target_device = device or self.tokens.weight.device\n        device = torch.device(\'cpu\')  # Build metadata on CPU; transfer once per batch.\n        rows, truncations = [], []\n        for event in events:\n            attrs = event.get(\'attributes\') or {}\n            if not isinstance(attrs, Mapping):\n                attrs = {\'raw\': attrs}\n            pairs = list(flatten(attrs))\n            truncated = max(0, len(pairs) - self.max_fields)\n            # Always retain type token; an empty event cannot create all-masked attention.\n            pairs = [(\'@event_type\', event.get(\'type\'))] + pairs[:self.max_fields]\n            rows.append(pairs)\n            truncations.append(truncated)\n        n = max((len(row) for row in rows), default=1)\n        keys,words,kinds,numeric,mask=[],[],[],[],[]\n        for row in rows:\n            rk,rw,rt,rn=[],[],[],[]\n            for key, value in row:\n                kind, parts, number = value_parts(value)\n                ids = [bucket(\'value:\'+p, self.buckets) for p in parts[:self.max_subtokens]]\n                rk.append(bucket(\'key:\'+key,self.buckets));rt.append(KINDS.index(kind))\n                rw.append(ids+[0]*(self.max_subtokens-len(ids)));rn.append([number])\n            padding=n-len(row)\n            keys.append(rk+[0]*padding);kinds.append(rt+[0]*padding)\n            words.append(rw+[[0]*self.max_subtokens]*padding)\n            numeric.append(rn+[[0.]]*padding);mask.append([False]*len(row)+[True]*padding)\n        arrays=(torch.tensor(keys,dtype=torch.long,device=device),\n                torch.tensor(words,dtype=torch.long,device=device),\n                torch.tensor(kinds,dtype=torch.long,device=device),\n                torch.tensor(numeric,dtype=torch.float32,device=device),\n                torch.tensor(mask,dtype=torch.bool,device=device))\n        return *(t.to(target_device) for t in arrays), truncations\n\n    def forward(self, events):\n        if not events:\n            return self.tokens.weight.new_empty((0,self.dim))\n        # The same observed event occurs in several endpoint history windows.\n        # Encode it once per batch, then gather; gradients from all uses sum.\n        # No stochastic event-encoder layers are used, so this preserves output.\n        unique=[];indices=[];seen={}\n        for event in events:\n            try:\n                identity=json.dumps({\'type\':event.get(\'type\'),\'attributes\':event.get(\'attributes\')},\n                                    sort_keys=True,separators=(\',\',\':\'))\n            except (TypeError,ValueError):\n                identity=None\n            index=seen.get(identity) if identity is not None else None\n            if index is None:\n                index=len(unique);unique.append(event)\n                if identity is not None:seen[identity]=index\n            indices.append(index)\n        key, word, kind, number, mask, _ = self.tokenize(unique)\n        valid = word.ne(0).unsqueeze(-1)\n        value = (self.tokens(word)*valid).sum(2)/valid.sum(2).clamp_min(1)\n        typed = self.kinds(kind) + self.numeric(number)\n        pairs = self.pair(torch.cat((self.tokens(key), value, typed), -1))\n        pooled, _ = self.attention(self.query.expand(len(unique),-1,-1), pairs, pairs,\n                                   key_padding_mask=mask, need_weights=False)\n        out=self.output(pooled[:,0])\n        return out[torch.tensor(indices,dtype=torch.long,device=out.device)]\n', 'sliding_node_mem_upgrade/encoding.py': '"""Cache only deterministic token IDs, never learned event embeddings."""\nfrom functools import lru_cache\nimport json\nimport numpy as np\nimport torch\nfrom orthrus_comparison.universal import UniversalEventEncoder, flatten, value_parts, bucket, KINDS\n\n\n@lru_cache(maxsize=32768)\ndef token_row(identity, buckets, max_fields, max_subtokens):\n    event=json.loads(identity); attrs=event.get(\'attributes\') or {}\n    if not isinstance(attrs,dict): attrs={\'raw\':attrs}\n    pairs=list(flatten(attrs)); truncated=max(0,len(pairs)-max_fields)\n    pairs=[(\'@event_type\',event.get(\'type\'))]+pairs[:max_fields]\n    keys=np.zeros(len(pairs),dtype=np.int64)\n    words=np.zeros((len(pairs),max_subtokens),dtype=np.int64)\n    kinds=np.zeros(len(pairs),dtype=np.int64); numeric=np.zeros((len(pairs),1),dtype=np.float32)\n    for j,(key,value) in enumerate(pairs):\n        kind,parts,number=value_parts(value)\n        keys[j]=bucket(\'key:\'+key,buckets); kinds[j]=KINDS.index(kind); numeric[j,0]=number\n        selected = parts if len(parts) <= max_subtokens else parts[:(max_subtokens+1)//2] + parts[-(max_subtokens//2):] if max_subtokens > 1 else parts[:1]\n        ids=[bucket(\'value:\'+p,buckets) for p in selected]\n        words[j,:len(ids)]=ids\n    return keys,words,kinds,numeric,truncated\n\n\nclass CachedUniversalEventEncoder(UniversalEventEncoder):\n    def tokenize(self, events, device=None):\n        try:\n            rows=[token_row(json.dumps({\'type\':e.get(\'type\'),\'attributes\':e.get(\'attributes\')},\n                                      sort_keys=True,separators=(\',\',\':\')),\n                            self.buckets,self.max_fields,self.max_subtokens) for e in events]\n        except (TypeError,ValueError):\n            return super().tokenize(events,device)\n        width=max((len(r[0]) for r in rows),default=1); count=len(rows)\n        keys=np.zeros((count,width),dtype=np.int64)\n        words=np.zeros((count,width,self.max_subtokens),dtype=np.int64)\n        kinds=np.zeros((count,width),dtype=np.int64)\n        numeric=np.zeros((count,width,1),dtype=np.float32)\n        mask=np.ones((count,width),dtype=bool)\n        for j,(k,w,t,n,_) in enumerate(rows):\n            size=len(k); keys[j,:size]=k; words[j,:size]=w; kinds[j,:size]=t; numeric[j,:size]=n; mask[j,:size]=False\n        target=device or self.tokens.weight.device\n        return *(torch.from_numpy(a).to(target) for a in (keys,words,kinds,numeric,mask)), [r[-1] for r in rows]\n\n\n@lru_cache(maxsize=32768)\ndef canonical_identity(event_type, attributes):\n    return json.dumps({\'type\': event_type, \'attributes\': json.loads(attributes)},\n                      sort_keys=True, separators=(\',\', \':\'))\n\n\nclass UpgradedUniversalEventEncoder(CachedUniversalEventEncoder):\n    """Pool only real subtokens, preserving their order with bounded weights.\n\n    Prepared queries carry canonical attributes, so observed relation variants\n    reuse the same serialization. No learned embedding is cached across steps.\n    """\n    def forward(self, events):\n        if not events:\n            return self.tokens.weight.new_empty((0, self.dim))\n        unique, indices, seen = [], [], {}\n        for event in events:\n            attributes = event.get(\'_canonical_attributes\')\n            if attributes is None:\n                try:\n                    attributes = json.dumps(event.get(\'attributes\'), sort_keys=True, separators=(\',\', \':\'))\n                except (TypeError, ValueError):\n                    return super().forward(events)\n            identity = (event.get(\'type\'), attributes)\n            if identity not in seen:\n                seen[identity] = len(unique)\n                unique.append(canonical_identity(identity[0], attributes))\n            indices.append(seen[identity])\n        rows = [token_row(i, self.buckets, self.max_fields, self.max_subtokens) for i in unique]\n        width = max(len(r[0]) for r in rows)\n        keys = np.zeros((len(rows), width), dtype=np.int64)\n        kinds = np.zeros_like(keys)\n        numeric = np.zeros((len(rows), width, 1), dtype=np.float32)\n        mask = np.ones((len(rows), width), dtype=bool)\n        words = np.zeros((len(rows), width, self.max_subtokens), dtype=np.int64)\n        for j, (key, word, kind, number, _) in enumerate(rows):\n            size = len(key)\n            keys[j, :size] = key; kinds[j, :size] = kind\n            numeric[j, :size] = number; mask[j, :size] = False\n            words[j, :size] = word\n        flat_words = words.reshape(-1, self.max_subtokens)\n        valid = flat_words != 0\n        counts = valid.sum(-1)\n        offsets = np.concatenate(([0], counts.cumsum())).astype(np.int64)\n        positions = np.arange(self.max_subtokens, dtype=np.float32)[None]\n        position_weights = (1. + positions / np.maximum(counts[:, None] - 1, 1)) * valid\n        position_weights /= np.maximum(position_weights.sum(-1, keepdims=True), 1.)\n        bags = flat_words[valid]\n        weights = position_weights[valid].astype(np.float32)\n        device = self.tokens.weight.device\n        def tensor(a):\n            return torch.as_tensor(a, device=\'cpu\').to(device, non_blocking=True)\n        value = torch.nn.functional.embedding_bag(\n            tensor(np.asarray(bags, dtype=np.int64)), self.tokens.weight,\n            tensor(np.asarray(offsets, dtype=np.int64)), mode=\'sum\', include_last_offset=True,\n            per_sample_weights=tensor(np.asarray(weights, dtype=np.float32)), padding_idx=0)\n        value = value.reshape(len(rows), width, self.dim)\n        typed = self.kinds(tensor(kinds)) + self.numeric(tensor(numeric))\n        pairs = self.pair(torch.cat((self.tokens(tensor(keys)), value, typed), -1))\n        pooled, _ = self.attention(self.query.expand(len(rows), -1, -1), pairs, pairs,\n                                  key_padding_mask=tensor(mask), need_weights=False)\n        return self.output(pooled[:, 0])[tensor(np.asarray(indices, dtype=np.int64))]\n', 'sliding_node_mem_v2/encoding.py': '"""Sequence-aware value encoding and bounded, weight-versioned inference cache."""\nfrom collections import OrderedDict\nfrom functools import lru_cache\nimport json\nimport numpy as np\nimport torch\nfrom sliding_node_mem_upgrade.encoding import UpgradedUniversalEventEncoder, token_row, canonical_identity\n\n\n@lru_cache(maxsize=32768)\ndef sequence_row(identity, buckets, max_fields, max_subtokens):\n    """Unigrams plus position-tagged adjacent subtokens, sharing the token table.\n\n    Unlike a weighted bag, this distinguishes equal positional-sum permutations.\n    Hashing is still lossy; this is a bounded local-order encoder, not lossless text.\n    """\n    keys, words, kinds, numbers, truncated = token_row(identity,buckets,max_fields,max_subtokens)\n    expanded = np.zeros((len(keys),2*max_subtokens),dtype=np.int64)\n    expanded[:,:max_subtokens] = words\n    if max_subtokens > 1:\n        left,right = words[:,:-1],words[:,1:]\n        positions = np.arange(1,max_subtokens,dtype=np.int64)[None]\n        valid = (left != 0) & (right != 0)\n        # The original tokenizer may retain the first and last half of a long\n        # value. Suppress the possible gap edge in all full-length rows.\n        full = (words != 0).sum(-1) == max_subtokens\n        valid[full,(max_subtokens+1)//2-1] = False\n        hashes = 1 + ((left*73856093) ^ (right*19349663) ^ (positions*83492791)) % (buckets-1)\n        expanded[:,max_subtokens:max_subtokens+max_subtokens-1] = np.where(valid,hashes,0)\n    return keys,expanded,kinds,numbers,truncated\n\n\nclass EventEncoder(UpgradedUniversalEventEncoder):\n    def __init__(self, *args, ordered_values=True, cache_size=8192, **kwargs):\n        super().__init__(*args, **kwargs)\n        self.ordered_values = ordered_values\n        self.cache_size = cache_size\n        self.clear_cache()\n\n    def clear_cache(self):\n        self._cache = OrderedDict()\n        self._cache_values = None\n        self._cache_version = None\n        self.cache_hits = self.cache_misses = 0\n\n    def train(self, mode=True):\n        if mode:\n            self.clear_cache()\n        return super().train(mode)\n\n    def _apply(self, fn, recurse=True):\n        self.clear_cache()\n        return super()._apply(fn, recurse=recurse)\n\n    def _load_from_state_dict(self, *args, **kwargs):\n        self.clear_cache()\n        return super()._load_from_state_dict(*args, **kwargs)\n\n    def encode(self, events):\n        identities = [canonical_identity(e.get(\'type\'),e.get(\'_canonical_attributes\') or\n                      json.dumps(e.get(\'attributes\'),sort_keys=True,separators=(\',\',\':\'))) for e in events]\n        row_fn = sequence_row if self.ordered_values else token_row\n        rows = [row_fn(key,self.buckets,self.max_fields,self.max_subtokens) for key in identities]\n        width = max(len(row[0]) for row in rows)\n        n = len(rows); length = self.max_subtokens*(2 if self.ordered_values else 1)\n        keys=np.zeros((n,width),dtype=np.int64); kinds=np.zeros_like(keys)\n        number=np.zeros((n,width,1),dtype=np.float32); mask=np.ones((n,width),dtype=bool)\n        words=np.zeros((n,width,length),dtype=np.int64)\n        for i,(key,word,kind,numeric,_) in enumerate(rows):\n            size=len(key);keys[i,:size]=key;kinds[i,:size]=kind;number[i,:size]=numeric\n            words[i,:size]=word;mask[i,:size]=False\n        flat=words.reshape(-1,length); valid=flat!=0; counts=valid.sum(-1)\n        offsets=np.concatenate(([0],counts.cumsum())).astype(np.int64)\n        if self.ordered_values:\n            weights=valid.astype(np.float32)/np.maximum(counts[:,None],1)\n        else:\n            positions=np.arange(length,dtype=np.float32)[None]\n            weights=(1+positions/np.maximum(counts[:,None]-1,1))*valid\n            weights/=np.maximum(weights.sum(-1,keepdims=True),1.)\n        device=self.tokens.weight.device\n        def tensor(a):\n            return torch.as_tensor(a,device=\'cpu\').to(device,non_blocking=True)\n        values=torch.nn.functional.embedding_bag(tensor(flat[valid]),self.tokens.weight,tensor(offsets),\n            mode=\'sum\',include_last_offset=True,per_sample_weights=tensor(weights[valid].astype(np.float32)),padding_idx=0)\n        values=values.reshape(n,width,self.dim)\n        typed=self.kinds(tensor(kinds))+self.numeric(tensor(number))\n        pairs=self.pair(torch.cat((self.tokens(tensor(keys)),values,typed),-1))\n        pooled,_=self.attention(self.query.expand(n,-1,-1),pairs,pairs,\n                                key_padding_mask=tensor(mask),need_weights=False)\n        return self.output(pooled[:, 0])\n\n    def forward(self, events):\n        if not events:\n            return self.tokens.weight.new_empty((0, self.dim))\n        identities, unique, positions, index = [], [], {}, []\n        for event in events:\n            # Prepared attributes are immutable by contract, as in QueryCache.\n            attributes = event.get(\'_canonical_attributes\')\n            if attributes is None:\n                attributes = json.dumps(event.get(\'attributes\'), sort_keys=True, separators=(\',\', \':\'))\n            identity = (event.get(\'type\'), attributes)\n            if identity not in positions:\n                positions[identity] = len(unique)\n                identities.append(identity)\n                unique.append(event)\n            index.append(positions[identity])\n        device = self.tokens.weight.device\n        gather = torch.tensor(index, device=device)\n        if self.training or torch.is_grad_enabled() or self.cache_size == 0:\n            return self.encode(unique)[gather]\n        version = tuple(p._version for p in self.parameters())\n        if version != self._cache_version:\n            self.clear_cache()\n            self._cache_version = version\n        if self._cache_values is None:\n            self._cache_values = self.tokens.weight.new_empty(self.cache_size, self.dim)\n        # Gather hits BEFORE inserting misses: eviction may reuse their slots.\n        hits, rows, missing = [], [], []\n        for i, identity in enumerate(identities):\n            row = self._cache.pop(identity, None)\n            if row is None:\n                missing.append(i)\n            else:\n                self._cache[identity] = row\n                hits.append(i)\n                rows.append(row)\n        values = self.tokens.weight.new_empty(len(unique), self.dim)\n        if hits:\n            values[torch.tensor(hits, device=device)] = self._cache_values[torch.tensor(rows, device=device)]\n        if missing:\n            new = self.encode([unique[i] for i in missing])\n            values[torch.tensor(missing, device=device)] = new\n            # One bulk store even if a single call has more misses than capacity.\n            destinations = {}\n            for j, i in enumerate(missing):\n                row = len(self._cache) if len(self._cache) < self.cache_size else self._cache.popitem(last=False)[1]\n                self._cache[identities[i]] = row\n                destinations[row] = j\n            rows = list(destinations)\n            self._cache_values[torch.tensor(rows, device=device)] = new[torch.tensor([destinations[r] for r in rows], device=device)]\n        self.cache_hits += len(hits)\n        self.cache_misses += len(missing)\n        return values[gather]\n', 'sliding_node_mem_v3/tokens.py': '"""Offline typed token rows and a vectorized memmap/resident token bank."""\nfrom array import array\nfrom collections import OrderedDict\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sqlite3\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nfrom orthrus_comparison.universal import flatten,value_parts,bucket,KINDS\nfrom sliding_node_mem_v2.encoding import EventEncoder\n\n\ndef hashes(text,buckets,two_hash):\n    first=bucket(text,buckets)\n    second=1+int.from_bytes(hashlib.blake2b(text.encode(),digest_size=8,person=b\'tgnn-hash-two\').digest(),\'big\')%(buckets-1)\n    return [first,second] if two_hash else [first]\n\n\ndef field_dtype(length,hash_count):\n    return np.dtype([(\'key\',\'<i4\',(hash_count,)),(\'word\',\'<i4\',(length,hash_count)),\n                     (\'kind\',\'<i8\'),(\'number\',\'<f4\')])\n\n\nclass TokenWriter:\n    def __init__(self,path,buckets=5303,max_fields=64,max_subtokens=32,\n                 two_hash=True,ordered=True):\n        self.path=Path(path);self.path.mkdir(parents=True,exist_ok=False)\n        self.config=dict(buckets=buckets,max_fields=max_fields,max_subtokens=max_subtokens,\n                         two_hash=two_hash,ordered=ordered)\n        self.dtype=field_dtype(max_subtokens*(2 if ordered else 1),2 if two_hash else 1)\n        self.file=(self.path/\'fields.bin\').open(\'wb\',buffering=1024*1024)\n        self.db=sqlite3.connect(self.path/\'identities.sqlite\')\n        self.db.execute(\'CREATE TABLE identities (identity TEXT PRIMARY KEY, idx INTEGER UNIQUE)\')\n        self.offsets=array(\'q\',[0]);self.cache=OrderedDict();self.max_width=1\n\n    def add(self,event):\n        identity=json.dumps({\'type\':event.get(\'type\'),\'attributes\':event.get(\'attributes\')},sort_keys=True,separators=(\',\',\':\'))\n        existing=self.cache.get(identity)\n        if existing is not None:\n            self.cache.move_to_end(identity);return existing\n        row=self.db.execute(\'SELECT idx FROM identities WHERE identity=?\',(identity,)).fetchone()\n        if row is not None:\n            index=row[0]\n        else:\n            index=len(self.offsets)-1\n            self.db.execute(\'INSERT INTO identities VALUES (?,?)\',(identity,index))\n            attrs=event.get(\'attributes\') or {}\n            if not isinstance(attrs,dict):attrs={\'raw\':attrs}\n            pairs=[(\'@event_type\',event.get(\'type\'))]+list(flatten(attrs))[:self.config[\'max_fields\']]\n            data=np.zeros(len(pairs),dtype=self.dtype)\n            length=self.config[\'max_subtokens\'];buckets=self.config[\'buckets\'];two=self.config[\'two_hash\']\n            for j,(key,value) in enumerate(pairs):\n                kind,parts,number=value_parts(value)\n                selected=parts if len(parts)<=length else parts[:(length+1)//2]+parts[-(length//2):] if length>1 else parts[:1]\n                tokens=[\'value:\'+p for p in selected]\n                if self.config[\'ordered\']:\n                    tokens += [f\'pair:{i}:{json.dumps([selected[i-1],selected[i]],ensure_ascii=False)}\'\n                               for i in range(1,len(selected)) if not (len(parts)>length and i==(length+1)//2)]\n                data[\'key\'][j]=hashes(\'key:\'+key,buckets,two)\n                data[\'word\'][j,:len(tokens)]=[hashes(p,buckets,two) for p in tokens]\n                data[\'kind\'][j]=KINDS.index(kind);data[\'number\'][j]=number\n            self.file.write(data.tobytes());self.offsets.append(self.offsets[-1]+len(data))\n            self.max_width=max(self.max_width,len(data))\n        self.cache[identity]=index\n        if len(self.cache)>65536:self.cache.popitem(last=False)\n        return index\n\n    def close(self):\n        self.file.close();self.db.commit();self.db.close()\n        np.save(self.path/\'offsets.npy\',np.asarray(self.offsets,dtype=np.int64))\n        config=dict(self.config,max_width=self.max_width,identities=len(self.offsets)-1,fields=self.offsets[-1],format=1)\n        (self.path/\'tokens.json\').write_text(json.dumps(config,indent=2))\n\n\nclass TokenBank:\n    def __init__(self,path):\n        self.path=Path(path);self.config=json.loads((self.path/\'tokens.json\').read_text())\n        c=self.config\n        self.fields=np.memmap(self.path/\'fields.bin\',mode=\'r\',dtype=field_dtype(c[\'max_subtokens\']*(2 if c[\'ordered\'] else 1),2 if c[\'two_hash\'] else 1))\n        self.offsets=np.load(self.path/\'offsets.npy\',mmap_mode=\'r\')\n        self.resident=None\n\n    def to_device(self,device):\n        # Optional: fitting the bank on GPU eliminates the remaining identity\n        # transfer/memmap staging. Runtime JSON/tokenization is absent either way.\n        self.resident={k:torch.from_numpy(np.array(self.fields[k])).to(device) for k in self.fields.dtype.names}\n        self.resident[\'offsets\']=torch.from_numpy(np.array(self.offsets)).to(device)\n        return self\n\n    def fetch(self,identities,device):\n        width=self.config[\'max_width\']\n        if self.resident is not None:\n            offsets=self.resident[\'offsets\'];identities=identities.to(offsets.device)\n            start=offsets[identities];sizes=offsets[identities+1]-start\n            index=start[:,None]+torch.arange(width,device=offsets.device)\n            mask=torch.arange(width,device=offsets.device)[None]>=sizes[:,None]\n            index=index.clamp_max(len(self.fields)-1)\n            data={k:self.resident[k][index].to(device) for k in self.fields.dtype.names}\n            mask=mask.to(device)\n        else:\n            ids=identities.detach().cpu().numpy()\n            start=self.offsets[ids];sizes=self.offsets[ids+1]-start\n            index=np.minimum(start[:,None]+np.arange(width),len(self.fields)-1)\n            mask=torch.from_numpy(np.arange(width)[None]>=sizes[:,None]).to(device)\n            rows=self.fields[index]\n            data={k:torch.from_numpy(np.array(rows[k])).to(device) for k in self.fields.dtype.names}\n        for k in (\'key\',\'word\'):\n            data[k]=data[k].long().masked_fill(mask.reshape(mask.shape+(1,)*(data[k].ndim-2)),0)\n        data[\'kind\']=data[\'kind\'].masked_fill(mask,0)\n        data[\'number\']=data[\'number\'].masked_fill(mask,0)\n        return data,mask\n\n\nclass PackedEncoder(EventEncoder):\n    """Same learned typed-KV modules, with already-tokenized unique identities."""\n    def __init__(self,bank,dim=64):\n        super().__init__(dim=dim,buckets=bank.config[\'buckets\'],cache_size=0,\n                         max_fields=bank.config[\'max_fields\'],max_subtokens=bank.config[\'max_subtokens\'])\n        self.bank=bank\n        self._frozen_table=None\n        self._frozen_signature=None\n\n    def _weight_signature(self):\n        return tuple((id(p),p._version,p.device,p.dtype) for p in self.parameters())\n\n    def clear_frozen_table(self):\n        self._frozen_table=None\n        self._frozen_signature=None\n\n    def _apply(self,fn,recurse=True):\n        self.clear_frozen_table()\n        return super()._apply(fn,recurse=recurse)\n\n    def _load_from_state_dict(self,*args,**kwargs):\n        self.clear_frozen_table()\n        return super()._load_from_state_dict(*args,**kwargs)\n\n    def train(self,mode=True):\n        if mode:self.clear_frozen_table()\n        return super().train(mode)\n\n    @torch.no_grad()\n    def precompute(self,batch_size=1024,dtype=None):\n        """Explicit bounded-chunk eval preparation; table is not checkpointed.\n\n        Default dtype preserves precision. Half precision is an opt-in ablation.\n        Normal optimizer/load_state_dict updates invalidate the lookup.\n        """\n        if self.training:raise ValueError(\'Call eval() before precomputing identities\')\n        if batch_size<1:raise ValueError(\'Positive precompute batch size required\')\n        self.clear_frozen_table()\n        device=self.tokens.weight.device\n        table=torch.empty(self.bank.config[\'identities\'],self.dim,device=device,\n                          dtype=dtype or self.tokens.weight.dtype)\n        for start in range(0,len(table),batch_size):\n            ids=torch.arange(start,min(start+batch_size,len(table)),device=device)\n            table[start:start+len(ids)]=self._encode(ids).to(table.dtype)\n        self._frozen_table=table\n        self._frozen_signature=self._weight_signature()\n        return table.numel()*table.element_size()\n\n    def forward(self,identities):\n        if self._frozen_table is not None:\n            if self._frozen_signature!=self._weight_signature():self.clear_frozen_table()\n            elif not self.training and not torch.is_grad_enabled():\n                return self._frozen_table[identities].to(self.tokens.weight.dtype)\n        unique,inverse=torch.unique(identities,sorted=True,return_inverse=True)\n        return self._encode(unique)[inverse]\n\n    def _encode(self,unique):\n        with torch.profiler.record_function(\'token_bank_fetch\'):\n            data,mask=self.bank.fetch(unique,self.tokens.weight.device)\n        words=data[\'word\'];valid=words[...,0]!=0\n        counts=valid.sum(-1,keepdim=True)\n        weights=valid.to(self.tokens.weight.dtype)\n        if not self.bank.config[\'ordered\']:\n            positions=torch.arange(words.shape[-2],device=words.device)\n            weights=weights*(1+positions/(counts-1).clamp_min(1))\n        weights=weights/weights.sum(-1,keepdim=True).clamp_min(1)\n        # Summing two embeddings uses the SAME table; no extra token parameters.\n        # Embedding-bag avoids materializing [U,F,L,H,64].\n        flat=words.reshape(-1,words.shape[-2]*words.shape[-1])\n        weight=weights[...,None].expand_as(words).reshape_as(flat)\n        offsets=torch.arange(0,flat.numel()+1,flat.shape[1],device=words.device)\n        value=F.embedding_bag(flat.flatten(),self.tokens.weight,offsets,\n                              mode=\'sum\',per_sample_weights=weight.flatten(),include_last_offset=True,padding_idx=0)\n        value=value.reshape(len(unique),words.shape[1],self.dim)\n        key=self.tokens(data[\'key\']).sum(-2)\n        typed=self.kinds(data[\'kind\'])+self.numeric(data[\'number\'][...,None])\n        pairs=self.pair(torch.cat((key,value,typed),-1))\n        pooled,_=self.attention(self.query.expand(len(unique),-1,-1),pairs,pairs,key_padding_mask=mask,need_weights=False)\n        return self.output(pooled[:,0])\n', 'orthrus_comparison/__init__.py': '', 'sliding_node_mem_upgrade/__init__.py': '', 'sliding_node_mem_v2/__init__.py': '', 'sliding_node_mem_v3/__init__.py': '', 'encoder_sgns.py': '"""Train Word2Vec\'s skip-gram/negative-sampling objective on benign path text.\n\n    Fit the resulting word vectors into the model\'s shared hash table. This is\n    corpus pretraining, not a claim of using ORTHRUS\'s original Word2Vec weights.\n"""\nimport argparse\nfrom collections import Counter\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom encoder_hash import sha256\nfrom sliding_node_mem_v3.tokens import hashes\n\n\ndef pretrain(prepared,output,epochs=5,dim=64,window=3,negatives=5,max_vocab=100000,\n             batch=4096,seed=0,device=\'cpu\',projection_steps=200):\n    if min(epochs,dim,window,negatives,max_vocab,batch,projection_steps)<1:raise ValueError(\'Positive pretraining settings required\')\n    root=Path(prepared);manifest=json.loads((root/\'manifest.json\').read_text())\n    corpus=root/\'benign_paths.jsonl\'\n    if sha256(corpus)!=manifest[\'checksums\'][\'benign_paths.jsonl\']:raise ValueError(\'Training corpus checksum mismatch\')\n    config=json.loads((root/\'tokens/tokens.json\').read_text())\n    if sha256(root/\'tokens/tokens.json\')!=manifest[\'checksums\'][\'tokens/tokens.json\']:raise ValueError(\'Token configuration checksum mismatch\')\n    counts=Counter()\n    with corpus.open() as stream:\n        for line in stream:counts.update(json.loads(line))\n    vocabulary=counts.most_common(max_vocab)\n    if len(vocabulary)<2:raise ValueError(\'Need at least two training subtokens\')\n    lookup={word:i for i,(word,_) in enumerate(vocabulary)}\n    rng=np.random.default_rng(seed);torch.manual_seed(seed)\n    inputs=nn.Embedding(len(vocabulary),dim,device=device)\n    outputs=nn.Embedding(len(vocabulary),dim,device=device)\n    nn.init.uniform_(inputs.weight,-.5/dim,.5/dim);nn.init.zeros_(outputs.weight)\n    distribution=torch.tensor([count**.75 for _,count in vocabulary],device=device)\n    distribution/=distribution.sum()\n    optimizer=torch.optim.Adam([inputs.weight,outputs.weight],lr=.01)\n    history=[]\n    for epoch in range(epochs):\n        pending=[];total=0.;seen=0\n        def update(pairs):\n            ids=torch.tensor(pairs,dtype=torch.long,device=device)\n            left=inputs(ids[:,0]);right=outputs(ids[:,1])\n            sampled=torch.multinomial(distribution,len(ids)*negatives,replacement=True).reshape(len(ids),negatives)\n            loss=(F.softplus(-(left*right).sum(-1))+F.softplus((left[:,None]*outputs(sampled)).sum(-1)).sum(-1)).mean()\n            optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step()\n            return float(loss.detach())*len(ids)\n        with corpus.open() as stream:\n            for line in stream:\n                words=[lookup[w] for w in json.loads(line) if w in lookup]\n                for i,word in enumerate(words):\n                    radius=int(rng.integers(1,window+1))\n                    pending.extend((word,words[j]) for j in range(max(0,i-radius),min(len(words),i+radius+1)) if j!=i)\n                    if len(pending)>=batch:\n                        total+=update(pending);seen+=len(pending);pending=[]\n        if pending:total+=update(pending);seen+=len(pending)\n        if not seen:raise ValueError(\'Corpus contains no context pairs\')\n        history.append(total/seen)\n    # Summation hashes share weights, so directly copying vectors into both\n    # buckets would double them and mishandle collisions. Fit their sum instead.\n    table=nn.Embedding(config[\'buckets\'],dim,padding_idx=0,device=device)\n    ids=torch.tensor([hashes(\'value:\'+word,config[\'buckets\'],config[\'two_hash\']) for word,_ in vocabulary],device=device)\n    target=inputs.weight.detach()\n    touched=torch.unique(ids)\n    with torch.no_grad():table.weight[touched]=0\n    fitting=torch.optim.Adam([table.weight],lr=.02)\n    for _ in range(projection_steps):\n        chosen=torch.randint(len(vocabulary),(min(batch,len(vocabulary)),),device=device)\n        loss=F.mse_loss(table(ids[chosen]).sum(1),target[chosen])\n        fitting.zero_grad(set_to_none=True);loss.backward();fitting.step()\n    with torch.no_grad():table.weight[0].zero_()\n    result=dict(tokens=table.weight.detach().cpu(),token_config=config,source_manifest_sha256=sha256(root/\'manifest.json\'),\n                corpus_sha256=sha256(corpus),objective=\'Word2Vec SGNS, followed by shared-hash sum fitting\',\n                epochs=epochs,seed=seed,vocabulary=len(vocabulary),history=history,projection_mse=float(loss.detach()))\n    path=Path(output);path.parent.mkdir(parents=True,exist_ok=True);torch.save(result,path)\n    return {k:v for k,v in result.items() if k!=\'tokens\'}\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--prepared\',required=True);p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--epochs\',type=int,default=5);p.add_argument(\'--device\',default=\'cpu\')\n    print(json.dumps(pretrain(**vars(p.parse_args())),indent=2))\n', 'encoder_hash.py': "import hashlib\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, 'rb') as f:\n        for block in iter(lambda: f.read(1024*1024), b''):\n            h.update(block)\n    return h.hexdigest()\n", 'experiment.py': '"""Self-contained encoder pilot; bundled with unmodified V3 sources in Colab."""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport math\nimport platform\nfrom pathlib import Path\nimport random\nimport time\nimport urllib.request\nfrom collections import Counter\nfrom dataclasses import dataclass, asdict\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport sklearn\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix\n\nfrom sliding_node_mem_v3.tokens import TokenWriter, TokenBank, PackedEncoder\nfrom orthrus_comparison.universal import flatten, value_parts\nfrom encoder_sgns import pretrain\n\nFEATURES = \'\'\'duration protocol_type service flag src_bytes dst_bytes land wrong_fragment urgent\nhot num_failed_logins logged_in num_compromised root_shell su_attempted num_root\nnum_file_creations num_shells num_access_files num_outbound_cmds is_host_login\nis_guest_login count srv_count serror_rate srv_serror_rate rerror_rate srv_rerror_rate\nsame_srv_rate diff_srv_rate srv_diff_host_rate dst_host_count dst_host_srv_count\ndst_host_same_srv_rate dst_host_diff_srv_rate dst_host_same_src_port_rate\ndst_host_srv_diff_host_rate dst_host_serror_rate dst_host_srv_serror_rate\ndst_host_rerror_rate dst_host_srv_rerror_rate\'\'\'.split()\n# Pinned mirror/checksum used by scikit-learn\'s official dataset loader.\nSOURCE_URL = \'https://ndownloader.figshare.com/files/5976042\'\nDATA_SHA256 = \'8045aca0d84e70e622d1148d7df782496f6333bf6eb979a1b0837c42a9fd9561\'\n\n\n@dataclass\nclass Config:\n    root: str = \'/content/tgdetect_encoder_run\'\n    train: int = 5000\n    val: int = 1000\n    calibration: int = 2000\n    test: int = 4000\n    attack_fraction: float = .20\n    batch: int = 128\n    epochs: int = 3\n    pretrain_epochs: int = 1\n    seeds: tuple = (0, 1, 2)\n    split_seed: int = 20261012\n    bootstrap: int = 300\n    latency_repeats: int = 60\n    device: str = \'cuda\'\n    score_fields: int = 4\n    cms_width: int = 65536\n    memory_budget_gib: float = 15.0\n\n\nARMS = {\n    \'old\': {},\n    \'old_extra_updates\': {\'extra\': True},\n    \'rarity\': {\'rarity\': True},\n    \'interaction\': {\'interaction\': True},\n    \'interaction_extra_updates\': {\'interaction\': True, \'extra\': True},\n    \'interaction_masked\': {\'interaction\': True, \'masked\': True},\n    \'rarity_interaction\': {\'rarity\': True, \'interaction\': True},\n    \'rarity_interaction_extra_updates\': {\'rarity\': True, \'interaction\': True, \'extra\': True},\n    \'full\': {\'rarity\': True, \'interaction\': True, \'masked\': True},\n    \'full_contrastive\': {\'rarity\': True, \'interaction\': True, \'masked\': True, \'contrastive\': True},\n}\n\n\ndef save_json(path, value):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(value, indent=2, default=lambda x: x.item() if isinstance(x, np.generic) else str(x)))\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\nclass FieldCMS:\n    """New exact-string field sketch, separate from endpoint-signature counts."""\n    def __init__(self, width=65536, depth=4):\n        self.width, self.depth = width, depth\n        self.table = np.zeros((depth, width), dtype=np.int64)\n\n    def indices(self, key, value):\n        text = json.dumps([key, value], sort_keys=True, separators=(\',\', \':\'))\n        raw = hashlib.blake2b(text.encode(), digest_size=8*self.depth).digest()\n        return [int.from_bytes(raw[8*i:8*i+8], \'little\') % self.width for i in range(self.depth)]\n\n    def read(self, key, value):\n        return int(min(self.table[i, h] for i, h in enumerate(self.indices(key, value))))\n\n    def add(self, key, value):\n        for i, h in enumerate(self.indices(key, value)):\n            self.table[i, h] += 1\n\n\ndef load_data(cfg):\n    root = Path(cfg.root); root.mkdir(parents=True, exist_ok=True)\n    path = root / \'kddcup.data_10_percent.gz\'\n    if not path.exists():\n        temp = path.with_suffix(\'.download\')\n        urls = [SOURCE_URL, \'https://figshare.com/ndownloader/files/5976042\']\n        errors = []\n        for url in urls:\n            try:\n                with urllib.request.urlopen(url, timeout=90) as src, temp.open(\'wb\') as dst:\n                    while chunk := src.read(1024 * 1024):\n                        dst.write(chunk)\n                if digest(temp) != DATA_SHA256:\n                    raise ValueError(\'Dataset SHA256 does not match the pinned scikit-learn archive\')\n                pd.read_csv(temp, compression=\'gzip\', header=None, nrows=2)\n                temp.replace(path); break\n            except Exception as exc:\n                errors.append(f\'{url}: {exc}\')\n        else:\n            raise RuntimeError(\'Dataset download failed; no synthetic fallback. \' + \'; \'.join(errors))\n    if digest(path) != DATA_SHA256:\n        raise ValueError(\'Cached dataset checksum mismatch; remove the corrupt download and retry\')\n    frame = pd.read_csv(path, header=None, names=FEATURES + [\'attack\'], compression=\'gzip\')\n    save_json(root / \'download.json\', {\'url\': SOURCE_URL, \'sha256\': digest(path), \'rows\': len(frame)})\n    return frame\n\n\ndef prepare_data(cfg, frame=None):\n    """Split before fitting statistics. No exact encoder input crosses splits."""\n    root = Path(cfg.root); root.mkdir(parents=True, exist_ok=True)\n    frame = load_data(cfg) if frame is None else frame.copy()\n    inputs = [c for c in FEATURES if c != \'service\']\n    frame[\'malicious\'] = frame.attack.ne(\'normal.\').astype(int)\n    # Group by ALL encoder fields, excluding the reconstruction target.\n    group = frame.groupby(inputs, dropna=False, sort=False)\n    inconsistent = group[[\'service\', \'malicious\']].transform(\'nunique\').max(axis=1).gt(1)\n    clean = frame.loc[~inconsistent].drop_duplicates(inputs).copy()\n    rng = np.random.default_rng(cfg.split_seed)\n    benign = clean.index[clean.malicious.eq(0)].to_numpy(); rng.shuffle(benign)\n    attacks = clean.index[clean.malicious.eq(1)].to_numpy(); rng.shuffle(attacks)\n    nt_attack = round(cfg.test * cfg.attack_fraction); nt_benign = cfg.test - nt_attack\n    needed = cfg.train + cfg.val + cfg.calibration + nt_benign\n    if len(benign) < needed or len(attacks) < nt_attack:\n        raise ValueError(f\'Not enough unique records: benign={len(benign)}, attack={len(attacks)}\')\n    t = cfg.train; v = t + cfg.val; c = v + cfg.calibration\n    split_indices = {\'train\': benign[:t], \'val\': benign[t:v], \'cal\': benign[v:c],\n                     \'test\': np.r_[benign[c:c+nt_benign], attacks[:nt_attack]]}\n    rng.shuffle(split_indices[\'test\'])\n    # Preserve source-row order only for the training-prefix count construction.\n    split_indices[\'train\'].sort()\n    splits = {k: clean.loc[ix].copy() for k, ix in split_indices.items()}\n    sets = [set(pd.util.hash_pandas_object(s[inputs], index=False).tolist()) for s in splits.values()]\n    for i, a in enumerate(sets):\n        for b in sets[i+1:]:\n            assert a.isdisjoint(b), \'Encoder input overlap across splits\'\n    train = splits[\'train\']\n    service_values = [\'<UNK>\'] + sorted(train.service.unique().tolist())\n    service_lookup = {s: i for i, s in enumerate(service_values)}\n    # Predict each attribute using targets fitted ONLY on benign training.\n    target_specs = {}\n    for col in inputs:\n        if pd.api.types.is_numeric_dtype(train[col]):\n            cuts = np.unique(np.quantile(train[col].astype(float), np.linspace(0, 1, 17)[1:-1]))\n            target_specs[col] = {\'cuts\': cuts.tolist(), \'classes\': len(cuts) + 1}\n        else:\n            values = train[col].value_counts().index[:63].tolist()\n            target_specs[col] = {\'values\': values, \'classes\': len(values) + 1}\n    bank_path = root / \'tokens\'\n    if bank_path.exists():\n        raise FileExistsError(\'Use a fresh run directory; token bank already exists: \' + str(bank_path))\n    writer = TokenWriter(bank_path)\n    sketch = FieldCMS(cfg.cms_width); exact = Counter(); result = {}\n    corpus = []\n    train_keys = None\n    for split, data in splits.items():\n        ids, rarity, targets, target_unknown, labels, services, field_identity = [], [], [], [], [], [], []\n        for _, row in data.iterrows():\n            attrs = {key: row[key].item() if isinstance(row[key], np.generic) else row[key] for key in inputs}\n            fields = [(\'@event_type\', None)] + list(flatten(attrs))\n            keys = [key for key, _ in fields]\n            if train_keys is None:\n                train_keys = keys\n            assert keys == train_keys\n            ids.append(writer.add({\'attributes\': attrs}))\n            field_identity.append([json.dumps([key, value], sort_keys=True, separators=(\',\', \':\')) for key, value in fields])\n            rarity.append([math.log1p(sketch.read(key, value)) for key, value in fields])\n            tg, unknown = [0], [False]\n            for key, value in list(flatten(attrs)):\n                # Flattened scalar key: remove its length prefix.\n                col = key.split(\':\', 1)[1]; spec = target_specs[col]\n                if \'cuts\' in spec:\n                    tg.append(int(np.searchsorted(spec[\'cuts\'], float(value), side=\'right\'))); unknown.append(False)\n                else:\n                    lookup = {v: i+1 for i, v in enumerate(spec[\'values\'])}\n                    target = lookup.get(value, 0); tg.append(target); unknown.append(target == 0)\n            targets.append(tg); target_unknown.append(unknown)\n            services.append(service_lookup.get(row.service, 0)); labels.append(int(row.malicious))\n            if split == \'train\':\n                # Read ALL fields before committing the current record.\n                for key, value in fields:\n                    sketch.add(key, value)\n                    exact[json.dumps([key, value], sort_keys=True)] += 1\n                # Adapt original path-token SGNS to benign categorical telemetry.\n                words = []\n                for value in attrs.values():\n                    kind, parts, _ = value_parts(value)\n                    if kind not in (\'number\', \'bool\', \'missing\'):\n                        words.extend(parts)\n                if len(words) >= 2:\n                    corpus.append(words)\n        result[split] = {\'ids\': np.array(ids), \'rarity\': np.array(rarity, np.float32),\n                         \'targets\': np.array(targets), \'unknown\': np.array(target_unknown),\n                         \'service\': np.array(services), \'y\': np.array(labels),\n                         \'field_identity\': np.array(field_identity),\n                         \'family\': data.attack.to_numpy().astype(str)}\n    writer.close()\n    corpus_path = root / \'benign_paths.jsonl\'\n    corpus_path.write_text(\'\'.join(json.dumps(words) + \'\\n\' for words in corpus))\n    save_json(root / \'manifest.json\', {\'checksums\': {\'benign_paths.jsonl\': digest(corpus_path),\n              \'tokens/tokens.json\': digest(bank_path / \'tokens.json\')}})\n    errors = []\n    for raw, count in exact.items():\n        key, value = json.loads(raw); errors.append(sketch.read(key, value) - count)\n    stats = {\'raw_rows\': len(frame), \'conflicting_rows_removed\': int(inconsistent.sum()),\n             \'unique_inputs\': len(clean), \'split_sizes\': {k: len(v[\'ids\']) for k, v in result.items()},\n             \'test_attack_fraction\': float(result[\'test\'][\'y\'].mean()),\n             \'split_seed\': cfg.split_seed, \'split_hashes\': {k: hashlib.sha256(ix.tobytes()).hexdigest() for k, ix in split_indices.items()},\n             \'cms\': {\'width\': cfg.cms_width, \'depth\': 4, \'bytes\': sketch.table.nbytes,\n                     \'mean_overcount\': float(np.mean(errors)), \'max_overcount\': int(max(errors)),\n                     \'fraction_exact\': float(np.mean(np.array(errors) == 0))},\n             \'unseen_service_fraction\': {k: float(np.mean(v[\'service\'] == 0)) for k, v in result.items()},\n             \'field_target_unknown_fraction\': {k: float(v[\'unknown\'][:, 1:].mean()) for k, v in result.items()},\n             \'test_families\': dict(Counter(result[\'test\'][\'family\'])),\n             \'temporal_claim\': False, \'pretraining_corpus\': \'benign training categorical values only\'}\n    stats[\'rarity_latency_scope\'] = \'counts are prepared on CPU; encoder timing includes FiLM but excludes online counting\'\n    save_json(root / \'data_audit.json\', stats)\n    return result, train_keys, target_specs, len(service_values), stats\n\n\nclass CandidateEncoder(PackedEncoder):\n    def __init__(self, bank, rarity=False, interaction=False, **unused):\n        super().__init__(bank)\n        self.use_rarity, self.use_interaction = rarity, interaction\n        if rarity:\n            self.film = nn.Linear(64, 128)\n            nn.init.zeros_(self.film.weight); nn.init.zeros_(self.film.bias)\n        if interaction:\n            self.interact = nn.TransformerEncoderLayer(64, 4, dim_feedforward=128,\n                dropout=0., activation=\'gelu\', batch_first=True, norm_first=True)\n        self.mask_value = nn.Parameter(torch.zeros(64))\n        self.field_table = None\n        self.field_signature = None\n        self.compact = False\n        self.int8_tokens = None\n\n    def train(self, mode=True):\n        if mode:\n            self.field_table = None\n            self.field_signature = None\n            self.int8_tokens = None\n        return super().train(mode)\n\n    def _apply(self, fn, recurse=True):\n        self.field_table = None\n        self.field_signature = None\n        self.int8_tokens = None\n        return super()._apply(fn, recurse=recurse)\n\n    def embed(self, ids):\n        if self.int8_tokens is None:\n            return self.tokens(ids)\n        quant, scale = self.int8_tokens\n        return (quant[ids].float() * scale[ids]).to(self.tokens.weight.dtype)\n\n    def fields(self, identities):\n        if self.field_table is not None and self.field_signature != self._weight_signature():\n            self.field_table = None\n            self.field_signature = None\n        if self.field_table is not None and not self.training and not torch.is_grad_enabled():\n            p, k, refs, pads = self.field_table\n            return p[refs[identities]], k[refs[identities]], pads[identities]\n        data, pad = self.bank.fetch(identities, self.tokens.weight.device)\n        words = data[\'word\']; valid = words[..., 0] != 0\n        weights = valid.to(self.tokens.weight.dtype)\n        weights = weights / weights.sum(-1, keepdim=True).clamp_min(1)\n        flat = words.reshape(-1, words.shape[-2] * words.shape[-1])\n        w = weights[..., None].expand_as(words).reshape_as(flat)\n        if self.compact or self.int8_tokens is not None:\n            # Compact IDs still require dynamic packing here; timed explicitly.\n            valid_flat = flat != 0\n            sizes = valid_flat.sum(-1)\n            offsets = torch.cat((sizes.new_zeros(1), sizes.cumsum(0)))\n            chosen, chosen_w = flat[valid_flat], w[valid_flat]\n            if self.int8_tokens is None:\n                value = F.embedding_bag(chosen, self.tokens.weight, offsets, mode=\'sum\',\n                    per_sample_weights=chosen_w, include_last_offset=True, padding_idx=0)\n            else:\n                bags = torch.repeat_interleave(torch.arange(len(flat), device=flat.device), sizes)\n                value = self.tokens.weight.new_zeros(len(flat), 64)\n                value.index_add_(0, bags, self.embed(chosen) * chosen_w[:, None])\n        else:\n            offsets = torch.arange(0, flat.numel()+1, flat.shape[1], device=flat.device)\n            value = F.embedding_bag(flat.flatten(), self.tokens.weight, offsets, mode=\'sum\',\n                per_sample_weights=w.flatten(), include_last_offset=True, padding_idx=0)\n        value = value.reshape(len(identities), words.shape[1], 64)\n        key = self.embed(data[\'key\']).sum(-2)\n        typed = self.kinds(data[\'kind\']) + self.numeric(data[\'number\'][..., None])\n        return self.pair(torch.cat((key, value, typed), -1)), key, pad\n\n    def forward(self, identities, rarity=None, hide=None, drop=None, return_fields=False):\n        # Unique STATIC fields only: rarity/hide/drop belong to request occurrences.\n        unique, inverse = torch.unique(identities, sorted=True, return_inverse=True)\n        p, key, pad = self.fields(unique)\n        p, key, pad = p[inverse], key[inverse], pad[inverse]\n        if self.use_rarity:\n            gamma, beta = self.film(key).chunk(2, -1)\n            r = rarity[..., None].to(p.dtype)\n            p = p * (1 + .1 * gamma.tanh() * r) + .1 * beta.tanh() * r\n        if hide is not None:\n            # Replace ENTIRE field representation, including kind/number/rarity.\n            p = torch.where(hide[..., None], key + self.mask_value, p)\n        if drop is not None:\n            pad = pad | drop\n            pad[:, 0] = False  # Always retain the event-type field.\n            p = p.masked_fill(pad[..., None], 0)\n        if self.use_interaction:\n            p = self.interact(p, src_key_padding_mask=pad)\n        pooled, _ = self.attention(self.query.expand(len(identities), -1, -1), p, p,\n                                   key_padding_mask=pad, need_weights=False)\n        out = self.output(pooled[:, 0])\n        return (out, p) if return_fields else out\n\n\nclass PilotModel(nn.Module):\n    def __init__(self, bank, classes, field_classes, options):\n        super().__init__()\n        self.options = options\n        # Keep the actual original baseline class, not a replacement architecture.\n        self.encoder = CandidateEncoder(bank, **options) if any(options.get(k) for k in (\'rarity\', \'interaction\')) else PackedEncoder(bank)\n        self.decoder = nn.Linear(64, classes)\n        if options.get(\'masked\'):\n            self.mask_heads = nn.ModuleList(nn.Linear(64, n) for n in field_classes)\n\n    def encode(self, ids, rarity, **kwargs):\n        if isinstance(self.encoder, CandidateEncoder):\n            return self.encoder(ids, rarity, **kwargs)\n        return self.encoder(ids)\n\n    def forward(self, ids, rarity):\n        return self.decoder(self.encode(ids, rarity))\n\n    def masked_loss(self, ids, rarity, targets, positions):\n        hide = torch.zeros_like(targets, dtype=torch.bool)\n        hide.scatter_(1, positions[:, None], True)\n        _, fields = self.encode(ids, rarity, hide=hide, return_fields=True)\n        losses = fields.new_zeros(len(ids))\n        for pos in positions.unique().tolist():\n            rows = positions == pos\n            losses[rows] = F.cross_entropy(self.mask_heads[pos](fields[rows, pos]), targets[rows, pos], reduction=\'none\')\n        return losses\n\n\ndef batches(data, cfg, order=None):\n    order = np.arange(len(data[\'ids\'])) if order is None else order\n    for start in range(0, len(order), cfg.batch):\n        ix = order[start:start+cfg.batch]\n        yield {k: torch.as_tensor(data[k][ix], device=cfg.device) for k in (\'ids\', \'rarity\', \'targets\', \'service\')}\n\n\ndef sync(cfg):\n    if cfg.device.startswith(\'cuda\'):\n        torch.cuda.synchronize()\n\n\ndef score(model, data, cfg, positions, include_masked=True):\n    model.eval(); channels = []\n    with torch.inference_mode():\n        for batch in batches(data, cfg):\n            ce = F.cross_entropy(model(batch[\'ids\'], batch[\'rarity\']), batch[\'service\'], reduction=\'none\')\n            cols = [ce]\n            if include_masked and model.options.get(\'masked\'):\n                losses = []\n                for pos in positions:\n                    where = torch.full_like(batch[\'ids\'], pos)\n                    losses.append(model.masked_loss(batch[\'ids\'], batch[\'rarity\'], batch[\'targets\'], where))\n                cols.append(torch.stack(losses).mean(0))\n            channels.append(torch.stack(cols, -1).float().cpu().numpy())\n    return np.concatenate(channels)\n\n\ndef cdf_fusion(reference, values):\n    # Fixed equal-weight rank fusion; reference uses only benign calibration A.\n    columns = []\n    for j in range(reference.shape[1]):\n        ref = np.sort(reference[:, j])\n        columns.append(np.searchsorted(ref, values[:, j], side=\'right\') / (len(ref)+1))\n    return np.stack(columns, -1).mean(-1)\n\n\ndef calibrated_threshold(scores, fpr):\n    # Strict > comparison; conservative finite-sample order statistic.\n    n = len(scores); index = min(n-1, max(0, math.ceil((n+1)*(1-fpr))-1))\n    return float(np.sort(scores)[index])\n\n\ndef wilson_interval(successes, total, z=1.96):\n    if total == 0:\n        return [None, None]\n    p = successes/total; denominator = 1+z*z/total\n    center = (p+z*z/(2*total))/denominator\n    radius = z*math.sqrt(p*(1-p)/total+z*z/(4*total*total))/denominator\n    return [max(0., center-radius), min(1., center+radius)]\n\n\ndef metrics(y, values, calibration):\n    result = {\'ap\': float(average_precision_score(y, values)), \'roc_auc\': float(roc_auc_score(y, values)),\n              \'test_prevalence\': float(np.mean(y))}\n    for fpr in (.01, .05):\n        threshold = calibrated_threshold(calibration, fpr)\n        predicted = values > threshold\n        tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()\n        precision = tp/max(tp+fp, 1); recall = tp/max(tp+fn, 1)\n        result[f\'fpr_target_{fpr}\'] = {\'threshold\': threshold, \'tn\': int(tn), \'fp\': int(fp), \'fn\': int(fn), \'tp\': int(tp),\n            \'precision\': precision, \'recall\': recall, \'f1\': 2*precision*recall/max(precision+recall, 1e-12),\n            \'test_fpr\': fp/max(tn+fp, 1), \'false_alerts_per_1000_benign\': 1000*fp/max(tn+fp, 1),\n            \'test_fpr_wilson95\': wilson_interval(fp, tn+fp),\n            \'recall_wilson95\': wilson_interval(tp, tp+fn),\n            \'precision_wilson95\': wilson_interval(tp, tp+fp)}\n    ranked = np.argsort(-values, kind=\'stable\')\n    result[\'precision_at\'] = {str(k): float(y[ranked[:min(k, len(y))]].mean()) for k in (10, 30, 100)}\n    return result\n\n\ndef train_arm(cfg, data, bank, n_services, field_classes, initialization, arm, seed, positions):\n    out = Path(cfg.root)/\'results\'/f\'{arm}_s{seed}\'\n    out.mkdir(parents=True, exist_ok=False)\n    seed_all(seed); options = ARMS[arm]\n    # Construct common modules under a fixed RNG stream, then copy to every arm.\n    baseline = PilotModel(bank, n_services, field_classes, {})\n    seed_all(seed + 10000)\n    # Common added blocks get identical initial weights even if another arm\n    # inserts FiLM before the interaction layer and consumes different RNG draws.\n    prototype = PilotModel(bank, n_services, field_classes, ARMS[\'full\'])\n    model = PilotModel(bank, n_services, field_classes, options).to(cfg.device)\n    state = {k: v for k, v in prototype.state_dict().items() if k in model.state_dict()}\n    state.update(baseline.state_dict())\n    model.load_state_dict(state, strict=True)\n    del baseline, prototype\n    with torch.no_grad():\n        model.encoder.tokens.weight.copy_(initialization.to(cfg.device))\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)\n    if cfg.device.startswith(\'cuda\'):\n        torch.cuda.reset_peak_memory_stats()\n    history = []; steps = 0; best = math.inf\n    sync(cfg); started = time.perf_counter()\n    phases = [(\'pretrain\', cfg.pretrain_epochs)] if options.get(\'masked\') or options.get(\'extra\') else []\n    phases.append((\'task\', cfg.epochs))\n    for phase, epochs in phases:\n        for epoch in range(epochs):\n            model.train()\n            order = np.random.default_rng(seed + epoch + (1000 if phase == \'pretrain\' else 0)).permutation(len(data[\'train\'][\'ids\']))\n            loss_total = 0.; examples = 0\n            for step, batch in enumerate(batches(data[\'train\'], cfg, order)):\n                if phase == \'pretrain\' and options.get(\'masked\'):\n                    generator = torch.Generator(device=cfg.device).manual_seed(seed*100000 + epoch*1000 + step)\n                    where = torch.randint(1, len(field_classes), (len(batch[\'ids\']),), generator=generator, device=cfg.device)\n                    loss = model.masked_loss(batch[\'ids\'], batch[\'rarity\'], batch[\'targets\'], where).mean()\n                    if options.get(\'contrastive\'):\n                        drops = torch.rand(batch[\'targets\'].shape, generator=generator, device=cfg.device) < .15\n                        a = F.normalize(model.encode(batch[\'ids\'], batch[\'rarity\']), dim=-1)\n                        b = F.normalize(model.encode(batch[\'ids\'], batch[\'rarity\'], drop=drops), dim=-1)\n                        logits = a @ b.T / .1\n                        target = torch.arange(len(a), device=a.device)\n                        loss = loss + .1*(F.cross_entropy(logits, target)+F.cross_entropy(logits.T, target))/2\n                else:\n                    loss = F.cross_entropy(model(batch[\'ids\'], batch[\'rarity\']), batch[\'service\'])\n                optimizer.zero_grad(set_to_none=True); loss.backward()\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)\n                optimizer.step(); steps += 1\n                loss_total += float(loss.detach())*len(batch[\'ids\']); examples += len(batch[\'ids\'])\n            val = float(score(model, data[\'val\'], cfg, positions, include_masked=False)[:, 0].mean())\n            row = {\'phase\': phase, \'epoch\': epoch+1, \'train_loss\': loss_total/examples, \'benign_val_service_ce\': val}\n            history.append(row); print(arm, seed, row, flush=True)\n            if phase == \'task\' and val < best:\n                best = val\n                torch.save({\'model\': model.state_dict(), \'options\': options, \'epoch\': epoch+1, \'seed\': seed}, out/\'best.pt\')\n    sync(cfg); train_seconds = time.perf_counter()-started\n    state = torch.load(out/\'best.pt\', map_location=cfg.device, weights_only=False)\n    model.load_state_dict(state[\'model\']); model.eval()\n    cal_channels = score(model, data[\'cal\'], cfg, positions)\n    test_channels = score(model, data[\'test\'], cfg, positions)\n    half = len(cal_channels)//2\n    # Service-only retains raw CE for every arm (including old); fused is secondary.\n    primary = metrics(data[\'test\'][\'y\'], test_channels[:, 0], cal_channels[half:, 0])\n    cutoff = primary[\'fpr_target_0.01\'][\'threshold\']\n    family_recall = {}\n    for family in sorted(set(data[\'test\'][\'family\'])):\n        subset = data[\'test\'][\'family\'] == family\n        if not data[\'test\'][\'y\'][subset].any():\n            continue\n        n = int(subset.sum()); caught = int((test_channels[subset, 0] > cutoff).sum())\n        family_recall[family] = {\'n\': n, \'detected\': caught, \'recall\': caught/n,\n                                 \'wilson95\': wilson_interval(caught, n)}\n    fused_test = cdf_fusion(cal_channels[:half], test_channels)\n    fused_cal = cdf_fusion(cal_channels[:half], cal_channels[half:])\n    fused = metrics(data[\'test\'][\'y\'], fused_test, fused_cal)\n    np.savez_compressed(out/\'scores.npz\', y=data[\'test\'][\'y\'], service=test_channels[:, 0], fused=fused_test,\n                        channels=test_channels, calibration=cal_channels, family=data[\'test\'][\'family\'])\n    timing = benchmark(model, data[\'test\'], cfg, positions)\n    result = {\'arm\': arm, \'seed\': seed, \'primary_service\': primary, \'secondary_fused\': fused,\n              \'attack_family_recall_at_1pct_target\': family_recall,\n              \'history\': history, \'checkpoint_epoch\': state[\'epoch\'], \'optimizer_updates\': steps,\n              \'training_seconds_including_validation\': train_seconds,\n              \'parameter_count\': sum(p.numel() for p in model.parameters()),\n              \'peak_cuda_bytes\': torch.cuda.max_memory_allocated() if cfg.device.startswith(\'cuda\') else None,\n              \'latency\': timing, \'checkpoint_sha256\': digest(out/\'best.pt\')}\n    save_json(out/\'result.json\', result)\n    return result, model\n\n\ndef timed(fn, cfg, repeats=None):\n    repeats = cfg.latency_repeats if repeats is None else repeats\n    with torch.inference_mode():\n        for _ in range(5):\n            fn()\n        sync(cfg); samples = []\n        for _ in range(repeats):\n            sync(cfg); start = time.perf_counter(); fn(); sync(cfg)\n            samples.append((time.perf_counter()-start)*1000)\n    return {\'p50_ms\': float(np.median(samples)), \'p95_ms\': float(np.quantile(samples, .95)),\n            \'p99_ms\': float(np.quantile(samples, .99)), \'samples_ms\': samples,\n            \'scope\': \'synchronized wall time; prepared inputs resident; excludes preparation and batch wait\'}\n\n\ndef benchmark(model, data, cfg, positions):\n    model.eval(); results = {}\n    for size in (1, 32, 128):\n        size = min(size, len(data[\'ids\']))\n        batch = {k: torch.as_tensor(data[k][:size], device=cfg.device) for k in (\'ids\', \'rarity\', \'targets\', \'service\')}\n        ids, r = batch[\'ids\'], batch[\'rarity\']\n        encoder = timed(lambda: model.encode(ids, r), cfg)\n        def scoring():\n            ce = F.cross_entropy(model(ids, r), batch[\'service\'], reduction=\'none\')\n            if model.options.get(\'masked\'):\n                for pos in positions:\n                    ce = ce + model.masked_loss(ids, r, batch[\'targets\'], torch.full_like(ids, pos)) / len(positions)\n            return ce\n        full = timed(scoring, cfg)\n        results[str(size)] = {\'encoder\': encoder, \'service_and_masked_channels\': full,\n                              \'masked_forward_passes\': len(positions) if model.options.get(\'masked\') else 0}\n    return results\n\n\ndef bootstrap_delta(y, old, new, repeats=300, seed=123):\n    rng = np.random.default_rng(seed); groups = [np.flatnonzero(y == label) for label in (0, 1)]\n    diffs = []\n    for _ in range(repeats):\n        ix = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])\n        diffs.append(average_precision_score(y[ix], new[ix])-average_precision_score(y[ix], old[ix]))\n    return {\'delta_ap\': float(average_precision_score(y, new)-average_precision_score(y, old)),\n            \'ci95_low\': float(np.quantile(diffs, .025)), \'ci95_high\': float(np.quantile(diffs, .975)),\n            \'scope\': \'paired stratified record bootstrap; conditional on this test sample; not capture-level uncertainty\'}\n\n\ndef acceptance_check(bank, data, cfg):\n    """Runs on the selected Colab GPU before any experiment training."""\n    seed_all(2026)\n    old = PackedEncoder(bank).to(cfg.device).eval()\n    candidate = CandidateEncoder(bank).to(cfg.device).eval()\n    candidate.load_state_dict(old.state_dict(), strict=False)\n    ids = torch.as_tensor(data[\'train\'][\'ids\'][:8], device=cfg.device)\n    ids = torch.cat((ids, ids[:3]))\n    with torch.inference_mode():\n        reference = old(ids)\n        error = float((candidate(ids)-reference).abs().max())\n        candidate.compact = True\n        compact_error = float((candidate(ids)-reference).abs().max())\n    if max(error, compact_error) > 1e-4:\n        raise AssertionError(f\'Original-encoder parity failed on {cfg.device}: {error}, {compact_error}\')\n    result = {\'device\': cfg.device, \'wrapper_max_abs_error\': error,\n              \'compact_max_abs_error\': compact_error, \'tolerance\': 1e-4}\n    save_json(Path(cfg.root)/\'results/encoder_acceptance.json\', result)\n    return result\n\n\ndef summarize(cfg, results):\n    rows = []\n    for result in results:\n        metric = result[\'primary_service\']; op = metric[\'fpr_target_0.01\']\n        rows.append({\'arm\': result[\'arm\'], \'seed\': result[\'seed\'], \'ap\': metric[\'ap\'], \'roc_auc\': metric[\'roc_auc\'],\n                     \'f1\': op[\'f1\'], \'precision\': op[\'precision\'], \'recall\': op[\'recall\'], \'test_fpr\': op[\'test_fpr\'],\n                     \'fp\': op[\'fp\'], \'tp\': op[\'tp\'], \'fused_ap\': result[\'secondary_fused\'][\'ap\'],\n                     \'train_seconds\': result[\'training_seconds_including_validation\'],\n                     \'parameters\': result[\'parameter_count\'], \'encoder_p50_b128\': result[\'latency\'][str(min(128, cfg.test))][\'encoder\'][\'p50_ms\']})\n    table = pd.DataFrame(rows)\n    out = Path(cfg.root)/\'results\'; table.to_csv(out/\'runs.csv\', index=False)\n    summary = table.groupby(\'arm\').agg({c: [\'mean\', \'std\'] for c in (\'ap\', \'f1\', \'recall\', \'test_fpr\', \'fused_ap\', \'encoder_p50_b128\')})\n    summary.to_csv(out/\'seed_summary.csv\')\n    intervals = []\n    for seed in cfg.seeds:\n        base = np.load(out/f\'old_s{seed}\'/\'scores.npz\')\n        for arm in sorted(set(table.arm)-{\'old\'}):\n            candidate = np.load(out/f\'{arm}_s{seed}\'/\'scores.npz\')\n            assert np.array_equal(base[\'y\'], candidate[\'y\'])\n            intervals.append({\'arm\': arm, \'seed\': seed, **bootstrap_delta(base[\'y\'], base[\'service\'], candidate[\'service\'], cfg.bootstrap)})\n    save_json(out/\'paired_ap_intervals.json\', intervals)\n    return table, summary\n\n\ndef run(cfg=Config(), frame=None, arms=None):\n    root = Path(cfg.root)\n    if (root/\'results\'/\'runs.csv\').exists():\n        raise FileExistsError(\'Completed results exist; choose a new Config.root to rerun.\')\n    if cfg.device.startswith(\'cuda\') and not torch.cuda.is_available():\n        raise RuntimeError(\'Select a GPU runtime in Colab. CPU smoke runs require device="cpu" explicitly.\')\n    torch.set_num_threads(2)\n    if cfg.device.startswith(\'cuda\'):\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n        total_memory = torch.cuda.get_device_properties(0).total_memory\n        torch.cuda.set_per_process_memory_fraction(min(1., cfg.memory_budget_gib*2**30/total_memory))\n    root.mkdir(parents=True, exist_ok=True)\n    save_json(root/\'config.json\', asdict(cfg))\n    save_json(root/\'environment.json\', {\'torch\': torch.__version__, \'numpy\': np.__version__, \'pandas\': pd.__version__,\n              \'sklearn\': sklearn.__version__, \'python\': platform.python_version(), \'cuda\': torch.version.cuda,\n              \'gpu\': torch.cuda.get_device_name() if cfg.device.startswith(\'cuda\') else \'CPU smoke only\',\n              \'vram_bytes\': torch.cuda.get_device_properties(0).total_memory if cfg.device.startswith(\'cuda\') else None,\n              \'source_sha256\': digest(__file__)})\n    preparation_start = time.perf_counter()\n    data, keys, specs, classes, audit = prepare_data(cfg, frame)\n    audit[\'preparation_seconds_including_download\'] = time.perf_counter()-preparation_start\n    save_json(root/\'data_audit.json\', audit)\n    # A cheap control detects whether learning adds anything beyond frequency.\n    cal_rare = np.exp(-data[\'cal\'][\'rarity\'][:, 1:]).mean(1)\n    test_rare = np.exp(-data[\'test\'][\'rarity\'][:, 1:]).mean(1)\n    save_json(root/\'results/rarity_only_control.json\', {\n        \'definition\': \'mean 1/(1+count(key,value)); excludes synthetic event-type field; no neural training\',\n        \'metrics\': metrics(data[\'test\'][\'y\'], test_rare, cal_rare[len(cal_rare)//2:])})\n    bank = TokenBank(root/\'tokens\')\n    if cfg.device.startswith(\'cuda\'):\n        bank.to_device(cfg.device)\n    print(\'Encoder acceptance:\', acceptance_check(bank, data, cfg), flush=True)\n    field_classes = [1] + [specs[k.split(\':\', 1)[1]][\'classes\'] for k in keys[1:]]\n    # Fixed field selection: training-only entropy; no test labels choose fields.\n    entropies = []\n    for i in range(1, len(keys)):\n        counts = np.bincount(data[\'train\'][\'targets\'][:, i], minlength=field_classes[i]); p = counts[counts>0]/counts.sum()\n        entropies.append((-float(-(p*np.log(p)).sum()), i))\n    positions = [i for _, i in sorted(entropies)[:cfg.score_fields]]\n    save_json(root/\'masked_score_fields.json\', {\'positions\': positions, \'keys\': [keys[i] for i in positions],\n              \'selection\': \'highest benign-training target entropy; fixed across all arms\'})\n    init_path = root/\'initial_embeddings.pt\'\n    init_started = time.perf_counter()\n    init_info = pretrain(root, init_path, epochs=1, batch=2048, projection_steps=30, seed=cfg.split_seed, device=cfg.device)\n    sync(cfg)\n    save_json(root/\'shared_sgns.json\', {**init_info, \'seconds\': time.perf_counter()-init_started,\n                                     \'sha256\': digest(init_path), \'shared_across_all_arms\': True})\n    init = torch.load(init_path, map_location=\'cpu\', weights_only=False)[\'tokens\']\n    results = []; last_models = {}\n    arms = list(ARMS) if arms is None else arms\n    for seed in cfg.seeds:\n        # Rotate arm order to reduce systematic thermal/startup bias.\n        order = arms[seed % len(arms):] + arms[:seed % len(arms)]\n        for arm in order:\n            try:\n                result, model = train_arm(cfg, data, bank, classes, field_classes, init, arm, seed, positions)\n            except Exception as exc:\n                save_json(root/\'results\'/f\'{arm}_s{seed}_failure.json\', {\'error\': repr(exc)})\n                raise\n            results.append(result)\n            if seed == cfg.seeds[-1] and arm in (\'old\', \'full\'):\n                last_models[arm] = model.cpu()\n            else:\n                del model\n            if cfg.device.startswith(\'cuda\'):\n                torch.cuda.empty_cache()\n    table, summary = summarize(cfg, results)\n    print(summary.to_string(), flush=True)\n    return {\'config\': cfg, \'data\': data, \'bank\': bank, \'models\': last_models, \'positions\': positions,\n            \'table\': table, \'summary\': summary, \'audit\': audit}\n', 'optimizations.py': '"""Optional measured execution variants and schema coverage probes."""\nimport copy\nimport math\nfrom pathlib import Path\nimport time\nfrom collections import Counter\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom experiment import (CandidateEncoder, PilotModel, batches, sync, timed, save_json,\n                        score, metrics, cdf_fusion, seed_all)\n\n\ndef wrapper(model):\n    device = model.encoder.tokens.weight.device\n    candidate = CandidateEncoder(model.encoder.bank,\n        rarity=model.options.get(\'rarity\', False), interaction=model.options.get(\'interaction\', False)).to(device)\n    candidate.load_state_dict(model.encoder.state_dict(), strict=False)\n    candidate.eval()\n    return candidate\n\n\n@torch.no_grad()\ndef field_cache(encoder, cfg):\n    """Deduplicate exact prepared field records, then store static learned fields."""\n    bank = encoder.bank; device = encoder.tokens.weight.device\n    # This is an explicit frozen-weight cache. Caller owns invalidation/rebuild.\n    start = time.perf_counter(); encoder.field_table = None\n    p_parts, k_parts, pads = [], [], []\n    n = bank.config[\'identities\']\n    for lo in range(0, n, cfg.batch):\n        ids = torch.arange(lo, min(lo+cfg.batch, n), device=device)\n        p, k, pad = encoder.fields(ids)\n        p_parts.append(p); k_parts.append(k); pads.append(pad)\n    p_all, k_all, pads = torch.cat(p_parts), torch.cat(k_parts), torch.cat(pads)\n    offsets = np.array(bank.offsets); sizes = np.diff(offsets)\n    owners = np.repeat(np.arange(n), sizes)\n    within = np.arange(len(bank.fields)) - np.repeat(offsets[:-1], sizes)\n    raw = np.array(bank.fields)\n    _, first, inverse = np.unique(raw.view(np.dtype((np.void, raw.dtype.itemsize))).ravel(),\n                                  return_index=True, return_inverse=True)\n    owner = torch.as_tensor(owners[first], device=device)\n    pos = torch.as_tensor(within[first], device=device)\n    # One extra zero entry for padding.\n    p_table = torch.cat((p_all.new_zeros(1, 64), p_all[owner, pos]))\n    k_table = torch.cat((k_all.new_zeros(1, 64), k_all[owner, pos]))\n    width = bank.config[\'max_width\']\n    refs = torch.zeros((n, width), dtype=torch.long, device=device)\n    refs[torch.as_tensor(owners, device=device), torch.as_tensor(within, device=device)] = torch.as_tensor(inverse+1, device=device)\n    encoder.field_table = (p_table, k_table, refs, pads)\n    encoder.field_signature = encoder._weight_signature()\n    sync(cfg)\n    return {\'build_seconds\': time.perf_counter()-start,\n            \'unique_field_records\': len(first), \'stored_field_records\': len(bank.fields),\n            \'cache_bytes\': sum(t.numel()*t.element_size() for t in encoder.field_table),\n            \'cache_type\': \'complete frozen field table, not bounded LRU\',\n            \'invalidation\': \'weight signature checked; training/device changes clear cache\',\n            \'coverage\': 1.0, \'build_peak_includes_temporary_per_identity_fields\': True}\n\n\n@torch.no_grad()\ndef execution_study(run):\n    cfg, data, bank = run[\'config\'], run[\'data\'], run[\'bank\']\n    results = []\n    for arm, model in run[\'models\'].items():\n        model = model.to(cfg.device).eval()\n        reference_encoder = wrapper(model)\n        # Actual held-out identities; KDD has few exact repeats after deduplication.\n        for workload in (\'actual_unique_test\', \'synthetic_repeat_heavy\'):\n            n = min(128, len(data[\'test\'][\'ids\']))\n            source = np.arange(n) if workload == \'actual_unique_test\' else np.arange(n) % min(8, n)\n            ids = torch.as_tensor(data[\'test\'][\'ids\'][source], device=cfg.device)\n            rarity = torch.as_tensor(data[\'test\'][\'rarity\'][source], device=cfg.device)\n            reference = reference_encoder(ids, rarity).float()\n            def measure(name, encoder, fn=None, extra=None):\n                fn = fn or (lambda: encoder(ids, rarity))\n                with torch.inference_mode():\n                    error = float((fn().float()-reference).abs().max())\n                record = {\'arm\': arm, \'workload\': workload, \'variant\': name,\n                          \'max_abs_embedding_error\': error, **timed(fn, cfg), **(extra or {})}\n                results.append(record)\n                return record\n            measure(\'fp32_eager\', reference_encoder)\n            compact = wrapper(model); compact.compact = True\n            measure(\'compact_bags\', compact)\n            cached = wrapper(model); info = field_cache(cached, cfg)\n            measure(\'frozen_field_table\', cached, extra=info)\n            # Full-event caching stores this fixed snapshot only; dynamic rarity is forbidden.\n            start = time.perf_counter()\n            table = []\n            # Cache only test identities for this workload, not arbitrary bank identities.\n            all_ids = torch.as_tensor(data[\'test\'][\'ids\'], device=cfg.device)\n            all_r = torch.as_tensor(data[\'test\'][\'rarity\'], device=cfg.device)\n            for lo in range(0, len(all_ids), cfg.batch):\n                table.append(reference_encoder(all_ids[lo:lo+cfg.batch], all_r[lo:lo+cfg.batch]))\n            event_table = torch.cat(table); sync(cfg)\n            event_info = {\'build_seconds\': time.perf_counter()-start, \'cache_bytes\': event_table.numel()*event_table.element_size(),\n                          \'cache_type\': \'test-identity frozen snapshot table; not LRU\', \'coverage\': 1.,\n                          \'streaming_rarity_compatible\': False if model.options.get(\'rarity\') else True}\n            request = torch.as_tensor(source, device=cfg.device)\n            measure(\'frozen_event_snapshot\', reference_encoder, lambda: event_table[request], event_info)\n            if cfg.device.startswith(\'cuda\'):\n                def bf16():\n                    with torch.autocast(\'cuda\', dtype=torch.bfloat16):\n                        return reference_encoder(ids, rarity)\n                measure(\'bf16_autocast\', reference_encoder, bf16)\n                quantized = wrapper(model)\n                w = quantized.tokens.weight.detach()\n                scale = w.abs().amax(1, keepdim=True).clamp_min(1e-8)/127\n                quantized.int8_tokens = (torch.round(w/scale).clamp(-127, 127).to(torch.int8), scale)\n                measure(\'int8_token_storage_prototype\', quantized,\n                        extra={\'packed_token_bytes\': w.numel()+scale.numel()*scale.element_size(),\n                               \'fp32_reference_token_bytes\': w.numel()*w.element_size(),\n                               \'reference_fp32_table_still_resident\': True})\n                # Compile/capture only static field-cache -> FiLM -> interaction -> pool.\n                class StaticTail(nn.Module):\n                    def __init__(self, encoder):\n                        super().__init__(); self.encoder = encoder\n                    def forward(self, identity, r):\n                        p_table, k_table, refs, pads = self.encoder.field_table\n                        p, key, pad = p_table[refs[identity]], k_table[refs[identity]], pads[identity]\n                        if self.encoder.use_rarity:\n                            gamma, beta = self.encoder.film(key).chunk(2, -1)\n                            p = p*(1+.1*gamma.tanh()*r[..., None])+.1*beta.tanh()*r[..., None]\n                        if self.encoder.use_interaction:\n                            p = self.encoder.interact(p, src_key_padding_mask=pad)\n                        pooled, _ = self.encoder.attention(self.encoder.query.expand(len(identity), -1, -1), p, p,\n                                                           key_padding_mask=pad, need_weights=False)\n                        return self.encoder.output(pooled[:, 0])\n                tail = StaticTail(cached).eval()\n                try:\n                    start = time.perf_counter(); compiled = torch.compile(tail, fullgraph=True)\n                    compiled(ids, rarity); sync(cfg)\n                    measure(\'compiled_static_tail\', cached, lambda: compiled(ids, rarity),\n                            {\'compile_seconds\': time.perf_counter()-start})\n                except Exception as exc:\n                    results.append({\'arm\': arm, \'workload\': workload, \'variant\': \'compiled_static_tail\', \'error\': repr(exc)})\n                try:\n                    static_ids, static_r = ids.clone(), rarity.clone()\n                    side = torch.cuda.Stream(); side.wait_stream(torch.cuda.current_stream())\n                    with torch.cuda.stream(side):\n                        for _ in range(3):\n                            tail(static_ids, static_r)\n                    torch.cuda.current_stream().wait_stream(side)\n                    graph = torch.cuda.CUDAGraph()\n                    with torch.cuda.graph(graph):\n                        static_out = tail(static_ids, static_r)\n                    def replay():\n                        static_ids.copy_(ids); static_r.copy_(rarity); graph.replay()\n                        return static_out\n                    measure(\'cuda_graph_static_tail\', cached, replay,\n                            {\'scope_detail\': \'includes fixed input-buffer copies and replay; field-cache build excluded\'})\n                except Exception as exc:\n                    results.append({\'arm\': arm, \'workload\': workload, \'variant\': \'cuda_graph_static_tail\', \'error\': repr(exc)})\n            del compact, cached, event_table\n        model.cpu()\n    save_json(Path(cfg.root)/\'results/execution_variants.json\', results)\n    return results\n\n\ndef distill(run, epochs=2):\n    cfg, data = run[\'config\'], run[\'data\']\n    if \'full\' not in run[\'models\'] or \'old\' not in run[\'models\']:\n        raise ValueError(\'Run both old and full arms first.\')\n    teacher = run[\'models\'][\'full\'].to(cfg.device).eval()\n    student = copy.deepcopy(run[\'models\'][\'old\'], memo={id(run[\'bank\']): run[\'bank\']}).to(cfg.device)\n    optimizer = torch.optim.AdamW(student.parameters(), lr=3e-4)\n    history = []; best = math.inf; best_state = None\n    for epoch in range(epochs):\n        student.train(); total = 0.; count = 0\n        order = np.random.default_rng(cfg.split_seed+epoch).permutation(len(data[\'train\'][\'ids\']))\n        for b in batches(data[\'train\'], cfg, order):\n            with torch.no_grad():\n                target = teacher.encode(b[\'ids\'], b[\'rarity\'])\n            embedding = student.encode(b[\'ids\'], b[\'rarity\'])\n            loss = F.mse_loss(embedding, target) + F.cross_entropy(student.decoder(embedding), b[\'service\'])\n            optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()\n            total += float(loss.detach())*len(embedding); count += len(embedding)\n        val = float(score(student, data[\'val\'], cfg, run[\'positions\'])[:, 0].mean())\n        history.append({\'epoch\': epoch+1, \'loss\': total/count, \'val_service_ce\': val})\n        if val < best:\n            best = val; best_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}\n    student.load_state_dict(best_state); student.eval()\n    cal = score(student, data[\'cal\'], cfg, run[\'positions\'])[:, 0]\n    test = score(student, data[\'test\'], cfg, run[\'positions\'])[:, 0]\n    result = {\'history\': history, \'metrics\': metrics(data[\'test\'][\'y\'], test, cal[len(cal)//2:]),\n              \'seed_count\': 1, \'scope\': \'optional teacher/student pilot, not part of primary three-seed matrix\'}\n    torch.save(best_state, Path(cfg.root)/\'results/distilled_student.pt\')\n    save_json(Path(cfg.root)/\'results/distillation.json\', result)\n    teacher.cpu(); student.cpu()\n    return result\n\n\ndef idf_select(attributes, training_documents, limit=64):\n    frequency = Counter(key for doc in training_documents for key in doc)\n    n = len(training_documents)\n    # Deterministic tie breaking; transform uses training statistics only.\n    ranked = sorted(attributes, key=lambda k: (-math.log((n+1)/(frequency[k]+1)), k))\n    return {k: attributes[k] for k in ranked[:limit]}\n\n\ndef unordered_list_pool(values, embedding):\n    # Full list, no first-32 truncation. Only valid for schema-declared sets/bags.\n    ids = torch.as_tensor(values, dtype=torch.long, device=embedding.weight.device)\n    total = embedding(ids).sum(0) if len(ids) else embedding.weight.new_zeros(embedding.embedding_dim)\n    return torch.cat((total, total.new_tensor([math.log1p(len(values))])))\n\n\ndef schema_probes():\n    seed_all(123)\n    attrs = {f\'common_{i:02}\': i for i in range(70)}; attrs[\'zz_rare_key\'] = \'evidence\'\n    docs = [{f\'common_{i:02}\': i for i in range(70)} for _ in range(20)]\n    priority = idf_select(attrs, docs)\n    embedding = nn.Embedding(100, 8)\n    forward = unordered_list_pool(list(range(50)), embedding)\n    reverse = unordered_list_pool(list(reversed(range(50))), embedding)\n    changed = unordered_list_pool(list(range(49))+[99], embedding)\n    return {\'idf_retains_rare_key\': \'zz_rare_key\' in priority, \'retained_fields\': len(priority),\n            \'list_permutation_max_error\': float((forward-reverse).abs().max().detach()),\n            \'late_element_change_detected\': bool(not torch.allclose(forward, changed)),\n            \'coverage\': \'synthetic functional probes only; not KDD detection results\',\n            \'list_policy\': \'use only for schema-declared unordered lists; order may be meaningful otherwise\'}\n', 'rarity_followup.py': '"""Controlled rarity follow-up. No masked-pretraining recipe in this matrix."""\nimport copy\nimport hashlib\nimport json\nimport math\nimport platform\nfrom pathlib import Path\nimport time\nfrom dataclasses import dataclass, asdict\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom experiment import (Config, CandidateEncoder, PilotModel, PackedEncoder, TokenBank,\n    FieldCMS, prepare_data, load_data, pretrain, batches, seed_all, score, benchmark,\n    metrics, bootstrap_delta, save_json, digest, sync, acceptance_check)\n\n\n@dataclass\nclass FollowupConfig(Config):\n    root: str = \'/content/tgdetect_rarity_followup\'\n    epochs: int = 8\n    seeds: tuple = (0, 1, 2, 3, 4)\n    split_seeds: tuple = (20261013, 20261014)\n    reference_rows: int = 1000\n    bootstrap: int = 300\n    latency_repeats: int = 40\n\n\nMATRIX = {\n    \'old\': (\'zero\', False),\n    \'prefix_log_film\': (\'prefix_log\', False),\n    \'frozen_log_film\': (\'frozen_log\', False),\n    \'frozen_surprise_film\': (\'frozen_surprise\', False),\n    \'shuffled_surprise_film\': (\'shuffled_surprise\', False),\n    \'frozen_surprise_interaction\': (\'frozen_surprise\', True),\n}\n\n\ndef read_fields(sketch, rows):\n    return np.array([[sketch.read(*json.loads(pair)) for pair in record] for record in rows], dtype=np.float32)\n\n\ndef insert_fields(sketch, rows):\n    for record in rows:\n        for pair in record:\n            sketch.add(*json.loads(pair))\n\n\ndef count_variants(data, cfg):\n    """Disjoint reference records never enter optimizer or validation."""\n    rng = np.random.default_rng(cfg.split_seed)\n    n = len(data[\'train\'][\'ids\'])\n    if not 1 <= cfg.reference_rows < n:\n        raise ValueError(\'reference_rows must be positive and smaller than train\')\n    picked = rng.permutation(n)[:cfg.reference_rows]\n    task_ix = np.setdiff1d(np.arange(n), picked)  # retained source order\n    reference = data[\'train\'][\'field_identity\'][picked]\n    task_data = {split: {k: (v[task_ix] if split == \'train\' else v) for k, v in values.items()}\n                 for split, values in data.items()}\n    frozen = FieldCMS(cfg.cms_width); insert_fields(frozen, reference)\n    started = time.perf_counter()\n    reference_counts = {split: read_fields(frozen, values[\'field_identity\']) for split, values in task_data.items()}\n    read_seconds = time.perf_counter()-started\n    prefix = FieldCMS(cfg.cms_width); prefix.table[:] = frozen.table\n    prefix_train = []\n    for record in task_data[\'train\'][\'field_identity\']:\n        prefix_train.append(read_fields(prefix, [record])[0])\n        insert_fields(prefix, [record])\n    prefix_counts = {\'train\': np.asarray(prefix_train)}\n    prefix_counts.update({split: read_fields(prefix, values[\'field_identity\'])\n                          for split, values in task_data.items() if split != \'train\'})\n    # Bounded normalized novelty, not an exact probabilistic likelihood.\n    scale = math.log1p(cfg.reference_rows)\n    surprises = {split: np.clip((scale-np.log1p(count))/scale, 0, 1).astype(np.float32)\n                 for split, count in reference_counts.items()}\n    variants = {}\n    for name in (\'zero\', \'prefix_log\', \'frozen_log\', \'frozen_surprise\', \'shuffled_surprise\'):\n        variants[name] = {}\n        for split, values in task_data.items():\n            if name == \'zero\':\n                rarity = np.zeros_like(values[\'rarity\'])\n            elif name == \'prefix_log\':\n                rarity = np.log1p(prefix_counts[split])\n            elif name == \'frozen_log\':\n                rarity = np.log1p(reference_counts[split])\n            else:\n                rarity = surprises[split].copy()\n                if name == \'shuffled_surprise\':\n                    # Each field shuffled independently; same marginals, broken value association.\n                    generator = np.random.default_rng(cfg.split_seed + len(split))\n                    for field in range(1, rarity.shape[1]):\n                        generator.shuffle(rarity[:, field])\n            rarity = rarity.astype(np.float32)\n            rarity[:, 0] = 0  # Prevent count of synthetic event-type field exposing stream age.\n            variants[name][split] = {**values, \'rarity\': rarity}\n    audit = {\'reference_rows\': len(picked), \'task_rows\': len(task_ix),\n             \'reference_identity_sha256\': hashlib.sha256(data[\'train\'][\'ids\'][picked].tobytes()).hexdigest(),\n             \'task_identity_sha256\': hashlib.sha256(task_data[\'train\'][\'ids\'].tobytes()).hexdigest(),\n             \'reference_disjoint_from_optimizer\': True, \'synthetic_type_rarity_zero\': True,\n             \'reference_count_read_seconds\': read_seconds,\n             \'cpu_read_ms_per_record\': 1000*read_seconds/sum(len(v[\'ids\']) for v in task_data.values()),\n             \'latency_scope\': \'CPU count reads measured separately; no claim of integrated streaming latency\',\n             \'novelty_definition\': \'clip((log1p(reference_size)-log1p(count))/log1p(reference_size),0,1)\',\n             \'prefix_vs_frozen_means\': {split: {\'prefix_log\': float(variants[\'prefix_log\'][split][\'rarity\'][:, 1:].mean()),\n                                               \'frozen_log\': float(variants[\'frozen_log\'][split][\'rarity\'][:, 1:].mean())}\n                                       for split in task_data},\n             \'unseen_field_fraction\': {s: float((c[:, 1:] == 0).mean()) for s,c in reference_counts.items()}}\n    return variants, surprises, reference, audit\n\n\ndef tail_scale(reference, values):\n    """Benign median/IQR normalization: preserve scores beyond observed tail."""\n    median = np.median(reference)\n    q25, q75 = np.quantile(reference, [.25, .75])\n    scale = max(float(q75-q25), float(np.std(reference))*.1, 1e-6)\n    return (values-median)/scale\n\n\ndef family_metrics(y, family, values, calibration):\n    result = metrics(y, values, calibration)\n    threshold = result[\'fpr_target_0.01\'][\'threshold\']\n    by_family = {}\n    for name in sorted(set(family[y == 1])):\n        use = family == name\n        by_family[name] = {\'n\': int(use.sum()), \'recall\': float((values[use] > threshold).mean())}\n    non_neptune = (y == 1) & (family != \'neptune.\')\n    result[\'non_neptune_n\'] = int(non_neptune.sum())\n    result[\'non_neptune_recall\'] = float((values[non_neptune] > threshold).mean()) if non_neptune.any() else None\n    result[\'macro_family_recall\'] = float(np.mean([v[\'recall\'] for v in by_family.values()])) if by_family else None\n    result[\'family_recall\'] = by_family\n    return result\n\n\ndef initialize(bank, classes, arm, seed, tokens, device):\n    seed_all(seed)\n    baseline = PilotModel(bank, classes, [], {})\n    seed_all(seed+10000)\n    prototype = PilotModel(bank, classes, [], {\'rarity\': True, \'interaction\': True})\n    mode, interaction = MATRIX[arm]\n    options = {} if arm == \'old\' else {\'rarity\': True, \'interaction\': interaction}\n    model = PilotModel(bank, classes, [], options).to(device)\n    state = {k:v for k,v in prototype.state_dict().items() if k in model.state_dict()}\n    state.update(baseline.state_dict()); model.load_state_dict(state)\n    with torch.no_grad():\n        model.encoder.tokens.weight.copy_(tokens.to(device))\n    return model\n\n\ndef train_one(cfg, bank, data, surprise, classes, tokens, arm, seed):\n    out = Path(cfg.root)/\'results\'/f\'{arm}_s{seed}\'; out.mkdir(parents=True, exist_ok=False)\n    model = initialize(bank, classes, arm, seed, tokens, cfg.device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)\n    history = []; best = math.inf; steps = 0\n    if cfg.device.startswith(\'cuda\'):\n        torch.cuda.reset_peak_memory_stats()\n    sync(cfg); start = time.perf_counter()\n    for epoch in range(cfg.epochs):\n        model.train(); total = 0.; seen = 0\n        order = np.random.default_rng(seed*1000+epoch).permutation(len(data[\'train\'][\'ids\']))\n        for batch in batches(data[\'train\'], cfg, order):\n            loss = F.cross_entropy(model(batch[\'ids\'], batch[\'rarity\']), batch[\'service\'])\n            optimizer.zero_grad(set_to_none=True); loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)\n            optimizer.step(); steps += 1\n            total += float(loss.detach())*len(batch[\'ids\']); seen += len(batch[\'ids\'])\n        val = float(score(model, data[\'val\'], cfg, [], False)[:, 0].mean())\n        history.append({\'epoch\': epoch+1, \'train_ce\': total/seen, \'benign_val_ce\': val})\n        if val < best:\n            best = val; torch.save({\'model\': model.state_dict(), \'epoch\': epoch+1}, out/\'best.pt\')\n    sync(cfg); seconds = time.perf_counter()-start\n    state = torch.load(out/\'best.pt\', map_location=cfg.device, weights_only=False)\n    model.load_state_dict(state[\'model\']); model.eval()\n    cal = score(model, data[\'cal\'], cfg, [], False)[:,0]\n    test = score(model, data[\'test\'], cfg, [], False)[:,0]\n    half = len(cal)//2; y = data[\'test\'][\'y\']; family = data[\'test\'][\'family\']\n    primary = family_metrics(y, family, test, cal[half:])\n    # Same frozen reference rarity for ALL models, including old.\n    rarity_cal = surprise[\'cal\'][:,1:].mean(1); rarity_test = surprise[\'test\'][:,1:].mean(1)\n    fusion = {}\n    for weight in (.25, .5, 1.):\n        a = tail_scale(cal[:half], cal[half:])+weight*tail_scale(rarity_cal[:half], rarity_cal[half:])\n        b = tail_scale(cal[:half], test)+weight*tail_scale(rarity_cal[:half], rarity_test)\n        fusion[str(weight)] = family_metrics(y, family, b, a)\n    # Counterfactual does not retrain: quantify reliance on learned rarity input.\n    disabled = {k: {**v, \'rarity\': np.zeros_like(v[\'rarity\'])} for k,v in data.items()}\n    zero_test = score(model, disabled[\'test\'], cfg, [], False)[:,0]\n    zero_cal = score(model, disabled[\'cal\'], cfg, [], False)[:,0]\n    reliance = family_metrics(y, family, zero_test, zero_cal[half:])\n    timing = benchmark(model, data[\'test\'], cfg, [])\n    result = {\'arm\': arm, \'seed\': seed, \'split_seed\': cfg.split_seed, \'primary\': primary,\n              \'predeclared_late_fusion\': fusion, \'zero_rarity_counterfactual\': reliance,\n              \'epochs\': cfg.epochs, \'updates\': steps, \'checkpoint_epoch\': state[\'epoch\'],\n              \'history\': history, \'training_seconds\': seconds, \'latency\': timing,\n              \'peak_cuda_bytes\': torch.cuda.max_memory_allocated() if cfg.device.startswith(\'cuda\') else None,\n              \'parameters\': sum(p.numel() for p in model.parameters()), \'checkpoint_sha256\': digest(out/\'best.pt\')}\n    np.savez_compressed(out/\'scores.npz\', y=y, family=family, service=test, calibration=cal,\n                        rarity_test=rarity_test, rarity_calibration=rarity_cal, zero_rarity=zero_test,\n                        zero_rarity_calibration=zero_cal)\n    save_json(out/\'result.json\', result)\n    print(arm, seed, \'AP\', round(primary[\'ap\'],4), \'F1\', round(primary[\'fpr_target_0.01\'][\'f1\'],4), flush=True)\n    del model, optimizer\n    return result\n\n\ndef run_followup(cfg=FollowupConfig(), frame=None):\n    if cfg.device.startswith(\'cuda\') and not torch.cuda.is_available():\n        raise RuntimeError(\'Choose GPU in Colab, or explicitly set device="cpu" for smoke tests.\')\n    root = Path(cfg.root); root.mkdir(parents=True, exist_ok=True)\n    if (root/\'runs.csv\').exists():\n        raise FileExistsError(\'Completed study exists. Choose a fresh root.\')\n    torch.set_num_threads(2)\n    if cfg.device.startswith(\'cuda\'):\n        torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False\n        total = torch.cuda.get_device_properties(0).total_memory\n        torch.cuda.set_per_process_memory_fraction(min(1.,cfg.memory_budget_gib*2**30/total))\n    save_json(root/\'config.json\', asdict(cfg))\n    save_json(root/\'environment.json\', {\'gpu\': torch.cuda.get_device_name() if cfg.device.startswith(\'cuda\') else \'CPU smoke\',\n              \'torch\': torch.__version__, \'cuda\': torch.version.cuda, \'python\': platform.python_version(),\n              \'source_sha256\': digest(__file__)})\n    frame = load_data(cfg) if frame is None else frame\n    rows = []; results = []; paired = []\n    for split_seed in cfg.split_seeds:\n        split_cfg = copy.copy(cfg); split_cfg.split_seed = split_seed\n        split_cfg.root = str(root/f\'split_{split_seed}\')\n        original, keys, specs, classes, audit = prepare_data(split_cfg, frame)\n        variants, surprise, reference, count_audit = count_variants(original, split_cfg)\n        save_json(Path(split_cfg.root)/\'count_protocol.json\', count_audit)\n        # Rebuild SGNS corpus from task records only (no validation/test).\n        from orthrus_comparison.universal import value_parts\n        words = []\n        for record in variants[\'zero\'][\'train\'][\'field_identity\']:\n            sequence = []\n            for raw in record:\n                _, value = json.loads(raw); kind, parts, _ = value_parts(value)\n                if kind not in (\'number\',\'bool\',\'missing\'):\n                    sequence.extend(parts)\n            if len(sequence)>1:\n                words.append(sequence)\n        corpus = Path(split_cfg.root)/\'benign_paths.jsonl\'\n        corpus.write_text(\'\'.join(json.dumps(w)+\'\\n\' for w in words))\n        manifest = Path(split_cfg.root)/\'manifest.json\'\n        value = json.loads(manifest.read_text()); value[\'checksums\'][\'benign_paths.jsonl\'] = digest(corpus)\n        save_json(manifest, value)\n        bank = TokenBank(Path(split_cfg.root)/\'tokens\')\n        if cfg.device.startswith(\'cuda\'):\n            bank.to_device(cfg.device)\n        acceptance_check(bank, variants[\'zero\'], split_cfg)\n        init_path = Path(split_cfg.root)/\'initial_embeddings.pt\'\n        init = pretrain(split_cfg.root, init_path, epochs=1, batch=2048, projection_steps=30,\n                        seed=split_seed, device=cfg.device)\n        save_json(Path(split_cfg.root)/\'shared_sgns.json\', init)\n        tokens = torch.load(init_path, map_location=\'cpu\', weights_only=False)[\'tokens\']\n        half = len(surprise[\'cal\'])//2\n        raw = family_metrics(original[\'test\'][\'y\'], original[\'test\'][\'family\'], surprise[\'test\'][:,1:].mean(1),\n                             surprise[\'cal\'][half:,1:].mean(1))\n        save_json(Path(split_cfg.root)/\'results/raw_rarity.json\', raw)\n        inverse_test = np.exp(-variants[\'frozen_log\'][\'test\'][\'rarity\'][:,1:]).mean(1)\n        inverse_cal = np.exp(-variants[\'frozen_log\'][\'cal\'][\'rarity\'][:,1:]).mean(1)\n        inverse = family_metrics(original[\'test\'][\'y\'], original[\'test\'][\'family\'], inverse_test, inverse_cal[half:])\n        save_json(Path(split_cfg.root)/\'results/raw_inverse_count.json\', inverse)\n        for seed in cfg.seeds:\n            order = list(MATRIX); shift = seed % len(order); order = order[shift:]+order[:shift]\n            for arm in order:\n                try:\n                    result = train_one(split_cfg, bank, variants[MATRIX[arm][0]], surprise, classes, tokens, arm, seed)\n                except Exception as exc:\n                    save_json(root/\'failure.json\', {\'split\': split_seed, \'seed\': seed, \'arm\': arm, \'error\': repr(exc)})\n                    raise\n                results.append(result); m=result[\'primary\']; op=m[\'fpr_target_0.01\']\n                rows.append({\'arm\':arm,\'seed\':seed,\'split_seed\':split_seed,\'ap\':m[\'ap\'],\'f1\':op[\'f1\'],\n                             \'recall\':op[\'recall\'],\'fpr\':op[\'test_fpr\'],\'non_neptune_recall\':m[\'non_neptune_recall\'],\n                             \'macro_family_recall\':m[\'macro_family_recall\'],\'checkpoint_epoch\':result[\'checkpoint_epoch\'],\n                             \'encoder_ms\':result[\'latency\'][str(min(128,cfg.test))][\'encoder\'][\'p50_ms\']})\n            for new, control in [(\'prefix_log_film\',\'old\'),(\'frozen_log_film\',\'prefix_log_film\'),\n                                 (\'frozen_surprise_film\',\'frozen_log_film\'),(\'frozen_surprise_film\',\'shuffled_surprise_film\'),\n                                 (\'frozen_surprise_interaction\',\'frozen_surprise_film\')]:\n                a=np.load(Path(split_cfg.root)/f\'results/{control}_s{seed}/scores.npz\')\n                b=np.load(Path(split_cfg.root)/f\'results/{new}_s{seed}/scores.npz\')\n                paired.append({\'split_seed\':split_seed,\'seed\':seed,\'candidate\':new,\'control\':control,\n                               **bootstrap_delta(a[\'y\'],a[\'service\'],b[\'service\'],cfg.bootstrap)})\n        del bank, tokens\n        if cfg.device.startswith(\'cuda\'):\n            torch.cuda.empty_cache()\n    table = pd.DataFrame(rows);table.to_csv(root/\'runs.csv\',index=False)\n    summary=table.groupby(\'arm\')[[\'ap\',\'f1\',\'recall\',\'fpr\',\'non_neptune_recall\',\'macro_family_recall\',\'encoder_ms\']].agg([\'mean\',\'std\'])\n    summary.to_csv(root/\'summary.csv\');save_json(root/\'paired_comparisons.json\',paired)\n    print(summary.to_string(),flush=True)\n    return table, summary\n', 'rarity_stability.py': '"""Longer-training and count robustness study; two encoder architectures."""\nimport copy\nimport hashlib\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn import functional as F\n\nfrom experiment import (TokenBank, FieldCMS, prepare_data, load_data, pretrain, batches,\n    score, metrics, bootstrap_delta, seed_all, save_json, digest, sync, timed)\nfrom rarity_followup import FollowupConfig, count_variants, initialize, family_metrics\n\n\n@dataclass\nclass StabilityConfig(FollowupConfig):\n    root: str = \'/content/tgdetect_rarity_stability\'\n    epochs: int = 16\n    budgets: tuple = (8, 12, 16)\n    split_seeds: tuple = (20261015, 20261016)\n    timing_repeats: int = 20\n\n\ndef fingerprints(rows):\n    """Hash each distinct canonical field once; keep all four uint64 hashes."""\n    cache = {}; output = np.empty((*rows.shape, 4), dtype=np.uint64)\n    for i, record in enumerate(rows):\n        for j, raw in enumerate(record):\n            if raw not in cache:\n                # Same serialization and BLAKE2 as original FieldCMS.\n                key, value = json.loads(raw)\n                text = json.dumps([key,value],sort_keys=True,separators=(\',\',\':\'))\n                cache[raw] = np.frombuffer(hashlib.blake2b(text.encode(),digest_size=32).digest(),dtype=\'<u8\').copy()\n            output[i,j] = cache[raw]\n    return output\n\n\ndef build_table(hashes, width):\n    table=np.zeros((4,width),dtype=np.int64)\n    positions=(hashes % np.uint64(width)).astype(np.int64)\n    for depth in range(4):\n        np.add.at(table[depth],positions[...,depth].ravel(),1)\n    return table\n\n\ndef lookup(table, hashes):\n    positions=(hashes % np.uint64(table.shape[1])).astype(np.int64)\n    result=table[np.arange(4),positions].min(-1)\n    return result\n\n\ndef gpu_lookup(table, positions):\n    counts=table[torch.arange(4,device=table.device),positions].amin(-1)\n    rarity=counts.float().log1p()\n    rarity[:,0]=0\n    return rarity\n\n\ndef count_conditions(data, reference, cfg):\n    started=time.perf_counter()\n    ref=fingerprints(reference)\n    hashes={split:fingerprints(values[\'field_identity\']) for split,values in data.items()}\n    compile_seconds=time.perf_counter()-started\n    clean=build_table(ref,cfg.cms_width)\n    conditions={\'clean\':clean,\'stale_half_reference\':build_table(ref[:max(1,len(ref)//2)],cfg.cms_width),\n                \'collision_small_table\':build_table(ref,2048)}\n    polluted=clean.copy()\n    # Simulated duplicated ingestion: repeat a random benign 30% of reference.\n    rng=np.random.default_rng(cfg.split_seed+99)\n    ix=rng.choice(len(ref),max(1,int(.3*len(ref))),replace=False)\n    polluted += build_table(ref[ix],cfg.cms_width)\n    conditions[\'duplicate_updates_30pct\']=polluted\n    results={}\n    for name,table in conditions.items():\n        results[name]={}\n        for split,values in data.items():\n            rarity=np.log1p(lookup(table,hashes[split])).astype(np.float32)\n            rarity[:,0]=0\n            results[name][split]={**values,\'rarity\':rarity}\n    results[\'missing_fields_10pct\']={}\n    for split,values in results[\'clean\'].items():\n        rng=np.random.default_rng(cfg.split_seed+len(split)+333)\n        rarity=values[\'rarity\'].copy();mask=rng.random(rarity.shape)<.1\n        rarity[mask]=0\n        results[\'missing_fields_10pct\'][split]={**values,\'rarity\':rarity}\n    audit={\'fingerprint_preparation_seconds\':compile_seconds,\n           \'fingerprint_bytes\':sum(h.nbytes for h in hashes.values())+ref.nbytes,\n           \'table_bytes\':clean.nbytes,\'frozen_reference_size\':len(ref),\n           \'synthetic_stress_conditions\':list(results),\n           \'missing_semantics\':\'zero log-count conflates unavailable counts and never-seen values; stress test only\'}\n    return results,hashes,clean,audit\n\n\ndef count_benchmark(cfg, data, hashes, table, model):\n    size=min(128,len(data[\'test\'][\'ids\']))\n    raw=data[\'test\'][\'field_identity\'][:size];hp=hashes[\'test\'][:size]\n    sketch=FieldCMS(table.shape[1]);sketch.table[:]=table\n    def python_counts():\n        counts=np.array([[sketch.read(*json.loads(pair)) for pair in row] for row in raw])\n        return counts\n    result={}\n    result[\'python_hash_and_lookup\']=timed(python_counts,cfg,cfg.timing_repeats)\n    result[\'numpy_prehashed_lookup\']=timed(lambda:lookup(table,hp),cfg,cfg.timing_repeats)\n    result[\'warm_identity_fingerprint_lookup\']=True\n    device=cfg.device\n    positions=torch.as_tensor((hp % np.uint64(table.shape[1])).astype(np.int64),device=device)\n    counts=torch.as_tensor(table,device=device)\n    ids=torch.as_tensor(data[\'test\'][\'ids\'][:size],device=device)\n    r=gpu_lookup(counts,positions)\n    reference=torch.as_tensor(np.log1p(lookup(table,hp)).astype(np.float32),device=device);reference[:,0]=0\n    torch.testing.assert_close(r,reference,atol=1e-6,rtol=1e-6)\n    result[\'device_prehashed_lookup\']=timed(lambda:gpu_lookup(counts,positions),cfg,cfg.timing_repeats)\n    result[\'encoder_prepared_rarity\']=timed(lambda:model.encode(ids,r),cfg,cfg.timing_repeats)\n    result[\'device_count_plus_encoder\']=timed(lambda:model.encode(ids,gpu_lookup(counts,positions)),cfg,cfg.timing_repeats)\n    host_positions=torch.as_tensor((hp % np.uint64(table.shape[1])).astype(np.int64))\n    host_ids=torch.as_tensor(data[\'test\'][\'ids\'][:size])\n    def staged():\n        p=host_positions.to(device);identity=host_ids.to(device)\n        return model.encode(identity,gpu_lookup(counts,p))\n    result[\'host_staging_count_plus_encoder\']=timed(staged,cfg,cfg.timing_repeats)\n    def deployed():\n        if model.options.get(\'rarity\'):\n            return staged()\n        return model.encoder(host_ids.to(device))\n    result[\'deployed_encoder_pipeline\']=timed(deployed,cfg,cfg.timing_repeats)\n    # Cold preprocessing is reported separately; no unseen-text work is hidden.\n    cold=[]\n    for _ in range(cfg.timing_repeats):\n        t=time.perf_counter();fingerprints(raw);cold.append((time.perf_counter()-t)*1000)\n    result[\'cold_field_fingerprints_ms\']={\'p50\':float(np.median(cold)),\'samples\':cold}\n    result[\'batch_records\']=size\n    result[\'scope\']=\'count+encoder only; no ingestion, service scoring, arrival wait, or state updates\'\n    return result\n\n\ndef train_stability(cfg, bank, data, conditions, hashes, table, classes, tokens, arm, seed):\n    out=Path(cfg.root)/\'results\'/f\'{arm}_s{seed}\';out.mkdir(parents=True,exist_ok=False)\n    model=initialize(bank,classes,arm,seed,tokens,cfg.device)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)\n    clean=conditions[\'clean\'];best=float(\'inf\');best_state=None;best_epoch=0;history=[];results=[]\n    if cfg.device.startswith(\'cuda\'):torch.cuda.reset_peak_memory_stats()\n    start=time.perf_counter()\n    for epoch in range(1,max(cfg.budgets)+1):\n        model.train();total=0.;n=0\n        order=np.random.default_rng(seed*1000+epoch-1).permutation(len(data[\'train\'][\'ids\']))\n        for batch in batches(clean[\'train\'],cfg,order):\n            loss=F.cross_entropy(model(batch[\'ids\'],batch[\'rarity\']),batch[\'service\'])\n            optimizer.zero_grad(set_to_none=True);loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(),1.);optimizer.step()\n            total+=float(loss.detach())*len(batch[\'ids\']);n+=len(batch[\'ids\'])\n        val=float(score(model,clean[\'val\'],cfg,[],False)[:,0].mean())\n        history.append({\'epoch\':epoch,\'train_ce\':total/n,\'benign_val_ce\':val})\n        if val<best:\n            best=val;best_epoch=epoch\n            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n        if epoch not in cfg.budgets:continue\n        # Evaluate the best benign checkpoint available within THIS budget.\n        current={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n        model.load_state_dict(best_state);model.eval()\n        prefix=out/f\'budget_{epoch}\';prefix.mkdir()\n        torch.save({\'model\':best_state,\'epoch\':best_epoch},prefix/\'best.pt\')\n        cal=score(model,clean[\'cal\'],cfg,[],False)[:,0];half=len(cal)//2\n        y=data[\'test\'][\'y\'];families=data[\'test\'][\'family\'];records={}\n        for name,values in conditions.items():\n            test=score(model,values[\'test\'],cfg,[],False)[:,0]\n            shifted_cal=cal if name==\'clean\' else score(model,values[\'cal\'],cfg,[],False)[:,0]\n            # Report unchanged deployed threshold and benign-only recalibration separately.\n            fixed=family_metrics(y,families,test,cal[half:])\n            recalibrated=family_metrics(y,families,test,shifted_cal[half:])\n            records[name]={\'fixed_clean_threshold\':fixed,\'benign_recalibrated\':recalibrated}\n            np.savez_compressed(prefix/f\'{name}_scores.npz\',y=y,family=families,service=test,\n                                clean_calibration=cal,condition_calibration=shifted_cal)\n        timing=count_benchmark(cfg,data,hashes,table,model)\n        sync(cfg)\n        result={\'arm\':arm,\'seed\':seed,\'split_seed\':cfg.split_seed,\'budget_epochs\':epoch,\n                \'checkpoint_epoch\':best_epoch,\'history\':history.copy(),\'conditions\':records,\n                \'count_latency\':timing,\'parameters\':sum(p.numel() for p in model.parameters()),\n                \'peak_cuda_bytes\':torch.cuda.max_memory_allocated() if cfg.device.startswith(\'cuda\') else None,\n                \'elapsed_seconds_including_previous_evaluation\':time.perf_counter()-start,\n                \'checkpoint_sha256\':digest(prefix/\'best.pt\')}\n        save_json(prefix/\'result.json\',result);results.append(result)\n        print(arm,seed,\'budget\',epoch,\'selected\',best_epoch,\'AP\',records[\'clean\'][\'fixed_clean_threshold\'][\'ap\'],flush=True)\n        model.load_state_dict(current)\n    return results\n\n\ndef run_stability(cfg=StabilityConfig(), frame=None):\n    if cfg.device.startswith(\'cuda\') and not torch.cuda.is_available():raise RuntimeError(\'Select GPU in Colab.\')\n    root=Path(cfg.root);root.mkdir(parents=True,exist_ok=True)\n    if (root/\'runs.csv\').exists():raise FileExistsError(\'Use a fresh output root.\')\n    torch.set_num_threads(2)\n    if cfg.device.startswith(\'cuda\'):\n        total=torch.cuda.get_device_properties(0).total_memory\n        torch.cuda.set_per_process_memory_fraction(min(1.,cfg.memory_budget_gib*2**30/total))\n        torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    from dataclasses import asdict\n    save_json(root/\'config.json\',asdict(cfg));save_json(root/\'environment.json\',{\n        \'gpu\':torch.cuda.get_device_name() if cfg.device.startswith(\'cuda\') else \'CPU smoke\',\n        \'torch\':torch.__version__,\'source_sha256\':digest(__file__)})\n    frame=load_data(cfg) if frame is None else frame;rows=[];paired=[]\n    for split in cfg.split_seeds:\n        scfg=copy.copy(cfg);scfg.split_seed=split;scfg.root=str(root/f\'split_{split}\')\n        data,keys,specs,classes,audit=prepare_data(scfg,frame)\n        variants,_,reference,count_audit=count_variants(data,scfg);data=variants[\'frozen_log\']\n        conditions,hashes,table,vector_audit=count_conditions(data,reference,scfg)\n        for s in data:\n            np.testing.assert_allclose(data[s][\'rarity\'],conditions[\'clean\'][s][\'rarity\'],atol=1e-6)\n        save_json(Path(scfg.root)/\'count_protocol.json\',{**count_audit,**vector_audit})\n        # SGNS uses optimizer records only, matching prior follow-up protocol.\n        from orthrus_comparison.universal import value_parts\n        corpus=[]\n        for record in data[\'train\'][\'field_identity\']:\n            words=[]\n            for raw in record:\n                _,value=json.loads(raw);kind,parts,_=value_parts(value)\n                if kind not in (\'number\',\'bool\',\'missing\'):words.extend(parts)\n            if len(words)>1:corpus.append(words)\n        p=Path(scfg.root)/\'benign_paths.jsonl\';p.write_text(\'\'.join(json.dumps(w)+\'\\n\' for w in corpus))\n        manifest=Path(scfg.root)/\'manifest.json\';m=json.loads(manifest.read_text());m[\'checksums\'][\'benign_paths.jsonl\']=digest(p);save_json(manifest,m)\n        bank=TokenBank(Path(scfg.root)/\'tokens\')\n        if cfg.device.startswith(\'cuda\'):bank.to_device(cfg.device)\n        initial=Path(scfg.root)/\'initial_embeddings.pt\'\n        info=pretrain(scfg.root,initial,epochs=1,batch=2048,projection_steps=30,seed=split,device=cfg.device)\n        save_json(Path(scfg.root)/\'shared_sgns.json\',info)\n        tokens=torch.load(initial,map_location=\'cpu\',weights_only=False)[\'tokens\']\n        for seed in cfg.seeds:\n            for arm in ([\'old\',\'frozen_log_film\'] if seed%2==0 else [\'frozen_log_film\',\'old\']):\n                try:\n                    results=train_stability(scfg,bank,data,conditions,hashes,table,classes,tokens,arm,seed)\n                except Exception as exc:\n                    save_json(root/\'failure.json\',{\'split_seed\':split,\'arm\':arm,\'seed\':seed,\'error\':repr(exc)})\n                    raise\n                for r in results:\n                    for condition,values in r[\'conditions\'].items():\n                        for policy,metric in values.items():\n                            op=metric[\'fpr_target_0.01\'];f=metric[\'family_recall\']\n                            rows.append({\'arm\':arm,\'seed\':seed,\'split_seed\':split,\'budget\':r[\'budget_epochs\'],\n                                \'checkpoint_epoch\':r[\'checkpoint_epoch\'],\'condition\':condition,\'threshold_policy\':policy,\n                                \'ap\':metric[\'ap\'],\'f1\':op[\'f1\'],\'fpr\':op[\'test_fpr\'],\'non_neptune_recall\':metric[\'non_neptune_recall\'],\n                                \'back_recall\':f.get(\'back.\',{}).get(\'recall\'),\'warezclient_recall\':f.get(\'warezclient.\',{}).get(\'recall\'),\n                                \'deployed_encoder_ms\':r[\'count_latency\'][\'deployed_encoder_pipeline\'][\'p50_ms\']})\n            for budget in cfg.budgets:\n                base=Path(scfg.root)/\'results\'\n                a=np.load(base/f\'old_s{seed}/budget_{budget}/clean_scores.npz\')\n                b=np.load(base/f\'frozen_log_film_s{seed}/budget_{budget}/clean_scores.npz\')\n                paired.append({\'split_seed\':split,\'seed\':seed,\'budget\':budget,\n                               **bootstrap_delta(a[\'y\'],a[\'service\'],b[\'service\'],cfg.bootstrap)})\n    df=pd.DataFrame(rows);df.to_csv(root/\'runs.csv\',index=False)\n    save_json(root/\'paired_comparisons.json\',paired)\n    summary=df.groupby([\'arm\',\'budget\',\'condition\',\'threshold_policy\'])[[\'ap\',\'f1\',\'fpr\',\'non_neptune_recall\',\'back_recall\',\'warezclient_recall\']].agg([\'mean\',\'std\'])\n    summary.to_csv(root/\'summary.csv\');return df,summary\n'}
for relative,source in SOURCE_BUNDLE.items():
    p=BUNDLE/relative;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(source)
sys.path.insert(0,str(BUNDLE))
(BUNDLE/"source_hashes.json").write_text('{\n  "orthrus_comparison/universal.py": "f31af3ead3add5999506dcc2681185692111a5db5594961eef8b589fc180a58f",\n  "sliding_node_mem_upgrade/encoding.py": "b220fa25448f545d257ab67801869e1253760aa48629e1f624f9de3c55a0b4bd",\n  "sliding_node_mem_v2/encoding.py": "d936babdd54ecfb188cc2bf07f3ebd3589bc301b307e925d28f9fa0a94864b8e",\n  "sliding_node_mem_v3/tokens.py": "5c5adb3c73d04dba57dfe75e08b691473524dc822e7ee3ad34535c0d301ba1af",\n  "orthrus_comparison/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n  "sliding_node_mem_upgrade/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n  "sliding_node_mem_v2/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n  "sliding_node_mem_v3/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n  "encoder_sgns.py": "960375937a1b5d76a21ad3440747a5c18532bb4ffa6d35a04091c5f19128b2b8",\n  "encoder_hash.py": "e268c2191b3235a17f3946286ca220cf3c8bb8734928315b75949de3ff7277ef",\n  "experiment.py": "303e83f2b11f522d6a9f08015a8270d8ee908c5e30b30ac8d2c99272f3756b41",\n  "optimizations.py": "af68fc72e31264af91aa9e3fe662935e9a013c0e7f9e77e2f2819ce346987b46",\n  "rarity_followup.py": "dad2c2da9367392ccb8754b1393fcd6cf3247be16ab72bb8b7f0ee000de43ba6",\n  "rarity_stability.py": "84eb427a948e2f8d57bf546f077cf98b2b125a8ea11c1739e15eaa341acb303c"\n}')


## Fixed research questions

1. Does original vs rarity change after longer equal training? Check convergence
   curves and whether checkpoint selection still reaches the budget boundary.
2. Can prehashed vectorized count lookup reduce the measured Python bottleneck?
   Report cold fingerprint preparation, Python hash+lookup, NumPy prehashed lookup,
   device lookup, and host-staging+count+encoder separately. Match CPU/GPU counts.
   The original deployed path does not pay for unnecessary rarity lookup.
3. How does the chosen checkpoint behave under count shifts?

| Condition | Construction | Interpretation |
|---|---|---|
| clean | Frozen 1,000-record benign reference | Main comparison |
| stale_half_reference | Only half the reference retained | Reduced coverage; proxy for stale snapshot, not verified temporal drift |
| duplicate_updates_30pct | Repeat a random 30% of benign reference | Duplicate-ingestion counter noise |
| collision_small_table | Width 2,048 instead of 65,536 | More CMS collision overcounts |
| missing_fields_10pct | Randomly replace 10% field log counts with zero | Missing/count-zero ambiguity; no trained fallback yet |

Stress choices are fixed without test labels. These simulations do not establish
real deployment drift. The existing encoder is not trained on corrupted counts.
Report frozen-threshold and condition-specific benign recalibration separately.
Recalibration assumes benign data from that condition is actually available.
Back, Warezclient, non-Neptune and all per-family recalls remain explicit, with
sample counts in result JSON. Rare-family estimates are uncertain; no test-driven
threshold or checkpoint selection is allowed.


In [ ]:
from rarity_stability import StabilityConfig,run_stability
CFG=StabilityConfig()
# Optional debug only:
# CFG.train=256;CFG.reference_rows=64;CFG.val=64;CFG.calibration=128
# CFG.test=256;CFG.seeds=(0,);CFG.split_seeds=(20261015,);CFG.budgets=(1,2)
# CFG.latency_repeats=5;CFG.timing_repeats=3;CFG.bootstrap=10
print(CFG)
TABLE,SUMMARY=run_stability(CFG)
display(TABLE)
display(SUMMARY)


In [ ]:
import pandas as pd
display(pd.read_json(Path(CFG.root)/'paired_comparisons.json'))
print('Clean-condition summary:')
display(TABLE[TABLE.condition.eq('clean')].groupby(['arm','budget','threshold_policy'])[
    ['ap','f1','fpr','non_neptune_recall','back_recall','warezclient_recall','deployed_encoder_ms']].agg(['mean','std']))


## Timing limits and decision rules

The device lookup uses the same four BLAKE2 bucket fingerprints as the Python
count-min reference, and takes the minimum across four rows. Hashes are computed
once per distinct canonical field in each preprocessing call. New field strings
still need hashing. The cold fingerprint cost is measured separately.
Device inputs and the frozen count table consume GPU memory. Staging benchmarks
include host-to-device fingerprint/identity copies, but exclude raw-event ingestion,
GAT/CNN/memory, scoring, batch-formation wait and online counter updates. Counts
are frozen: this notebook does not claim a causal streaming update implementation.

Keep the original if rarity gains disappear with adequate training. If counts
improve minority-family recall but increase false alerts, compare operating points
using benign-only thresholds; do not report AP as proof of low-FPR superiority.
If realistic count shifts cause failure, test a separately trained missing-count
fallback/rarity dropout next. If both encoders still miss Back or Warezclient,
investigate missing features or the service-reconstruction objective on development
data before adding more layers. Budget checkpoints are correlated measurements
of the same training trajectory. Record bootstrap intervals are not independent
capture uncertainty. Do not choose the best tested budget and present it as an
untouched prospective evaluation.


In [ ]:
import shutil
shutil.copy(BUNDLE/'source_hashes.json',Path(CFG.root)/'source_hashes.json')
shutil.copytree(BUNDLE,Path(CFG.root)/'source_snapshot',dirs_exist_ok=True)
archive=shutil.make_archive('/content/tgdetect_rarity_stability_results','zip',CFG.root)
from google.colab import files
files.download(archive)
